# YOLO11 Custom Attention - A2 Threshold Sweep

This notebook sweeps confidence and NMS IoU thresholds for the retained custom-attention candidates.
The main selection metric is healthy-aware score, while full-test and diseased-only mask mAP are kept for auditability.

Default grid: `conf=[0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.50]`, `iou=[0.50,0.60,0.70]`.


### Dependency Setup


In [ ]:
# Install runtime dependency used by YOLO training/evaluation cells.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
print('Ultralytics dependency is ready.')


### Standalone Runtime and Custom Modules


In [ ]:

# ============================================================
# Standalone runtime for YOLO11n custom-attention notebooks
# This cell embeds helper code, custom attention modules, and YAMLs.
# You can upload/run this single notebook without the _shared folder.
# ============================================================
from pathlib import Path
import sys, os, types, inspect, importlib.util, subprocess, math
import pandas as pd
from IPython.display import display

_STANDALONE_GROUP = 'A'
_HELPER_SOURCES = {'A': '"""Evaluation and calibration helpers for custom-attention Group A.\n\nGroup A is intentionally inference/protocol focused:\n- A1: TTA inference at the default threshold.\n- A2: confidence/NMS threshold sweep.\n- A3: matched strong-augmentation baseline.\n\nThe helpers reuse the same split/evaluation ideas from the clean baseline and\nthe custom_attention_multiseed pipeline. They do not edit existing notebooks.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport glob\nimport inspect\nimport math\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n\nIMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")\nDEFAULT_IMGSZ = 640\nDEFAULT_CONF = 0.25\nDEFAULT_IOU = 0.70\nDEFAULT_CONF_LIST = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]\nDEFAULT_IOU_LIST = [0.50, 0.60, 0.70]\n\nCOUNT_PENALTY_WEIGHT = 0.05\nDISEASE_MISS_PENALTY_WEIGHT = 0.15\nHEALTHY_FP_PENALTY_WEIGHT = 0.10\n\nSTRONG_AUG_TRAIN_ARGS: dict[str, Any] = {\n    "auto_augment": None,\n    "erasing": 0.15,\n    "mosaic": 0.0,\n    "mixup": 0.0,\n    "cutmix": 0.0,\n    "copy_paste": 0.0,\n    "fliplr": 0.5,\n    "flipud": 0.3,\n    "hsv_h": 0.05,\n    "hsv_s": 0.50,\n    "hsv_v": 0.40,\n    "degrees": 10.0,\n    "translate": 0.10,\n    "scale": 0.50,\n    "shear": 0.0,\n    "perspective": 0.0,\n    "multi_scale": 0.0,\n    "bgr": 0.0,\n}\n\n\nDEFAULT_CANDIDATES: list[dict[str, Any]] = [\n    {\n        "key": "baseline_clean",\n        "display_name": "Baseline clean YOLO11n-seg",\n        "family": "baseline",\n        "augmentation": "clean_light",\n        "model_yaml": "yolo11n-seg.pt",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/segment/yolo11n-seg_shrimp_seg_clean_light_aug_baseline_clean_light_aug_baseline/weights/best.pt",\n            "/kaggle/working/runs/segment/*clean_light_aug_baseline*/weights/best.pt",\n        ],\n    },\n    {\n        "key": "triplet_attention_segment_head",\n        "display_name": "Triplet Attention Segment Head",\n        "family": "custom_attention",\n        "augmentation": "clean_light",\n        "model_yaml": "custom_attention/triplet_attention_segment_head/yolov11n_triplet_segment.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/segment/*triplet_attention_segment_head*/weights/best.pt",\n            "/kaggle/working/runs/custom_attention/*triplet_attention_segment_head*/weights/best.pt",\n            "/kaggle/working/runs/custom_attention_multiseed/triplet_attention_segment_head/seed_42/weights/best.pt",\n            "runs/custom_attention_multiseed/triplet_attention_segment_head/seed_42/weights/best.pt",\n        ],\n    },\n    {\n        "key": "cote_gate",\n        "display_name": "CoTE Gate",\n        "family": "custom_attention_research_modules",\n        "augmentation": "clean_light",\n        "model_yaml": "custom_attention_research_modules/cote_gate/cote_gate.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/custom_attention_research_modules/cote_gate/weights/best.pt",\n        ],\n    },\n    {\n        "key": "cote_gate_strong_augmentation",\n        "display_name": "CoTE Gate strong augmentation",\n        "family": "custom_attention_research_modules",\n        "augmentation": "strong",\n        "model_yaml": "custom_attention_research_modules/cote_gate/generated_yamls/cote_gate_strong_augmentation.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/custom_attention_research_modules/cote_gate_strong_augmentation/weights/best.pt",\n        ],\n    },\n    {\n        "key": "lpsc_gate",\n        "display_name": "LPSC Gate",\n        "family": "custom_attention_research_modules",\n        "augmentation": "clean_light",\n        "model_yaml": "custom_attention_research_modules/lpsc_gate/lpsc_gate.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/custom_attention_research_modules/lpsc_gate/weights/best.pt",\n        ],\n    },\n    {\n        "key": "lpsc_gate_strong_augmentation",\n        "display_name": "LPSC Gate strong augmentation",\n        "family": "custom_attention_research_modules",\n        "augmentation": "strong",\n        "model_yaml": "custom_attention_research_modules/lpsc_gate/generated_yamls/lpsc_gate_strong_augmentation.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/custom_attention_research_modules/lpsc_gate_strong_augmentation/weights/best.pt",\n        ],\n    },\n    {\n        "key": "cesa_lite_segment_head",\n        "display_name": "CESA-Lite Segment Head",\n        "family": "custom_attention",\n        "augmentation": "clean_light",\n        "model_yaml": "custom_attention/cesa_lite_segment_head/yolov11n_cesa_lite.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/segment/*cesa_lite_segment_head*/weights/best.pt",\n            "/kaggle/working/runs/custom_attention/*cesa_lite_segment_head*/weights/best.pt",\n            "/kaggle/working/runs/custom_attention_multiseed/cesa_lite_segment_head/seed_42/weights/best.pt",\n            "runs/custom_attention_multiseed/cesa_lite_segment_head/seed_42/weights/best.pt",\n        ],\n    },\n    {\n        "key": "sge_eca_head_gate",\n        "display_name": "SGE-ECA Head Gate",\n        "family": "custom_attention",\n        "augmentation": "clean_light",\n        "model_yaml": "custom_attention/sge_eca_head_gate/model.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/custom_attention/*sge_eca_head_gate*/weights/best.pt",\n            "/kaggle/working/runs/segment/*sge_eca_head_gate*/weights/best.pt",\n            "/kaggle/working/runs/custom_attention_multiseed/sge_eca_head_gate/seed_42/weights/best.pt",\n            "runs/custom_attention_multiseed/sge_eca_head_gate/seed_42/weights/best.pt",\n        ],\n    },\n]\n\n\ndef attention_root() -> Path:\n    """Return the yolov11n_attention root when this helper lives in the repo."""\n\n    here = Path(__file__).resolve()\n    for parent in [*here.parents, Path.cwd(), Path.cwd() / "yolov11n_attention"]:\n        if (parent / "custom_attention").exists() and (parent / "custom_attention_research_modules").exists():\n            return parent\n    return here.parents[2]\n\n\ndef ensure_project_import_paths() -> Path:\n    """Add vendored Ultralytics and attention module roots to sys.path."""\n\n    root = attention_root()\n    vendored = root / "ultralytics"\n    paths = []\n    if (vendored / "ultralytics" / "__init__.py").exists():\n        paths.append(vendored)\n    paths.extend([root, root.parent, Path.cwd()])\n    for path in paths:\n        text = str(path)\n        if text not in sys.path:\n            sys.path.insert(0, text)\n    return root\n\n\ndef register_all_custom_attention_modules() -> None:\n    """Register all custom attention modules needed to load checkpoints/YAMLs."""\n\n    ensure_project_import_paths()\n    import ultralytics.nn.tasks as tasks\n\n    from custom_attention.common.attention_modules import (\n        AttentionGate,\n        CALiteSpatialGate,\n        CESALite,\n        LowFPCBAMLite,\n        NAMAttention,\n        TripletAttention,\n    )\n    from custom_attention._shared.attention_modules import (\n        BoundaryAwareLiteAttention,\n        CASpatialLowFPGate,\n        CESLite,\n        ContextSuppressionGateLite,\n        LowFPResidualSpatialGate,\n        P3P4SemanticAttentionGate,\n        PrototypeAwareMaskGateLite,\n        SGEECAHeadGate,\n    )\n    from custom_attention_research_modules._shared.attention_modules import RESEARCH_ATTENTION_MODULES\n\n    single_input_modules = (\n        CESALite,\n        TripletAttention,\n        CALiteSpatialGate,\n        NAMAttention,\n        LowFPCBAMLite,\n        SGEECAHeadGate,\n        CESLite,\n        LowFPResidualSpatialGate,\n        P3P4SemanticAttentionGate,\n        BoundaryAwareLiteAttention,\n        ContextSuppressionGateLite,\n        CASpatialLowFPGate,\n        PrototypeAwareMaskGateLite,\n        *RESEARCH_ATTENTION_MODULES,\n    )\n    for module_cls in (*single_input_modules, AttentionGate):\n        setattr(tasks, module_cls.__name__, module_cls)\n        setattr(sys.modules["__main__"], module_cls.__name__, module_cls)\n\n    tasks.NHOMA_SINGLE_INPUT_ATTENTION_MODULES = single_input_modules\n    tasks.AttentionGate = AttentionGate\n\n    if not getattr(tasks, "_shrimp_nhoma_attention_parse_patched", False):\n        source = inspect.getsource(tasks.parse_model)\n        marker = "        elif m in frozenset(\\n            {\\n                Detect,"\n        insert = """        elif m in NHOMA_SINGLE_INPUT_ATTENTION_MODULES:\n            c1 = ch[f]\n            c2 = c1\n            args = [c1, *args]\n        elif m is AttentionGate:\n            c1 = ch[f[0]]\n            cg = ch[f[1]]\n            c2 = c1\n            args = [c1, cg, *args]\n"""\n        if marker not in source:\n            raise RuntimeError("Could not patch ultralytics.nn.tasks.parse_model: Detect marker not found.")\n        patched = source.replace(marker, insert + marker, 1)\n        exec(compile(patched, "<shrimp_nhoma_attention_parse_model>", "exec"), tasks.__dict__)\n        tasks._shrimp_nhoma_attention_parse_patched = True\n\n    _try_add_torch_safe_globals()\n\n\ndef _try_add_torch_safe_globals() -> None:\n    """Help newer torch versions unpickle checkpoints with custom classes."""\n\n    try:\n        import torch\n        import ultralytics.nn.tasks as tasks\n    except Exception:\n        return\n\n    names = {\n        candidate\n        for candidate in [\n            "AttentionGate",\n            "CALiteSpatialGate",\n            "CESALite",\n            "CoTEGate",\n            "LPSCGate",\n            "SGEECAHeadGate",\n            "TripletAttention",\n            "CESLite",\n            "ContextSuppressionGateLite",\n        ]\n        if hasattr(tasks, candidate)\n    }\n    classes = [getattr(tasks, name) for name in sorted(names)]\n    try:\n        torch.serialization.add_safe_globals(classes)\n    except Exception:\n        pass\n\n\ndef candidate_by_key(key: str) -> dict[str, Any]:\n    for candidate in DEFAULT_CANDIDATES:\n        if candidate["key"] == key:\n            return dict(candidate)\n    raise KeyError(f"Unknown candidate key: {key}")\n\n\ndef selected_candidates(module_keys: list[str] | None = None) -> list[dict[str, Any]]:\n    if module_keys is None:\n        return [dict(item) for item in DEFAULT_CANDIDATES]\n    requested = set(module_keys)\n    return [dict(item) for item in DEFAULT_CANDIDATES if item["key"] in requested]\n\n\ndef _path_roots() -> list[Path]:\n    root = attention_root()\n    roots = [Path.cwd(), root, root.parent]\n    unique: list[Path] = []\n    for item in roots:\n        if item not in unique:\n            unique.append(item)\n    return unique\n\n\ndef _candidate_pattern_strings(pattern: str | Path) -> list[str]:\n    text = str(pattern)\n    path = Path(text)\n    if path.is_absolute():\n        return [text]\n    return [str(root / text) for root in _path_roots()]\n\n\ndef resolve_existing_path(path_value: str | Path | None) -> Path | None:\n    if not path_value:\n        return None\n    for text in _candidate_pattern_strings(path_value):\n        path = Path(text)\n        if path.exists():\n            return path\n    return None\n\n\ndef glob_existing(pattern: str | Path) -> list[Path]:\n    matches: list[Path] = []\n    for text in _candidate_pattern_strings(pattern):\n        for value in glob.glob(text):\n            path = Path(value)\n            if path.exists() and path not in matches:\n                matches.append(path)\n    return sorted(matches, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)\n\n\ndef resolve_checkpoint(candidate: dict[str, Any], overrides: dict[str, str | Path] | None = None) -> Path | None:\n    overrides = overrides or {}\n    override = overrides.get(candidate["key"])\n    if override:\n        path = resolve_existing_path(override)\n        if path is None:\n            raise FileNotFoundError(f"Override checkpoint does not exist for {candidate[\'key\']}: {override}")\n        return path\n\n    for pattern in candidate.get("checkpoint_patterns", []):\n        matches = glob_existing(pattern)\n        if matches:\n            return matches[0]\n    return None\n\n\ndef checkpoint_table(\n    module_keys: list[str] | None = None,\n    overrides: dict[str, str | Path] | None = None,\n) -> list[dict[str, Any]]:\n    rows = []\n    for candidate in selected_candidates(module_keys):\n        checkpoint = resolve_checkpoint(candidate, overrides=overrides)\n        model_yaml = resolve_existing_path(candidate.get("model_yaml"))\n        rows.append(\n            {\n                "module_key": candidate["key"],\n                "display_name": candidate["display_name"],\n                "family": candidate.get("family"),\n                "augmentation": candidate.get("augmentation"),\n                "checkpoint": str(checkpoint) if checkpoint else "",\n                "checkpoint_found": checkpoint is not None,\n                "model_yaml": str(model_yaml or candidate.get("model_yaml", "")),\n                "model_yaml_found": bool(model_yaml) or str(candidate.get("model_yaml", "")).endswith(".pt"),\n            }\n        )\n    return rows\n\n\ndef validate_split_exists(dataset_dir: Path) -> None:\n    missing = []\n    for split in ("train", "valid", "test"):\n        for subdir in ("images", "labels"):\n            path = dataset_dir / split / subdir\n            if not path.exists():\n                missing.append(str(path))\n    if missing:\n        raise FileNotFoundError("Prepared grouped split is incomplete. Missing: " + ", ".join(missing))\n\n\ndef remove_yolo_label_caches(root: Path) -> None:\n    for cache_path in Path(root).glob("**/*.cache"):\n        cache_path.unlink()\n\n\ndef copy_dataset_snapshot(source_dataset: Path, snapshot_dir: Path, refresh: bool = False) -> Path:\n    validate_split_exists(source_dataset)\n    if snapshot_dir.exists() and refresh:\n        shutil.rmtree(snapshot_dir)\n    if snapshot_dir.exists():\n        validate_split_exists(snapshot_dir)\n        remove_yolo_label_caches(snapshot_dir)\n        return snapshot_dir\n    snapshot_dir.parent.mkdir(parents=True, exist_ok=True)\n    ignore = shutil.ignore_patterns("runs", "*.cache", ".clahe_applied")\n    shutil.copytree(source_dataset, snapshot_dir, ignore=ignore)\n    validate_split_exists(snapshot_dir)\n    remove_yolo_label_caches(snapshot_dir)\n    return snapshot_dir\n\n\ndef write_data_yaml(source_data_yaml: Path, dataset_dir: Path, yaml_path: Path, val_dir: str = "valid", test_dir: str = "test") -> Path:\n    if not source_data_yaml.exists():\n        raise FileNotFoundError(f"Source data YAML not found: {source_data_yaml}")\n    with source_data_yaml.open("r", encoding="utf-8") as f:\n        content = yaml.safe_load(f) or {}\n    content["train"] = str(dataset_dir / "train" / "images")\n    content["val"] = str(dataset_dir / val_dir / "images")\n    content["test"] = str(dataset_dir / test_dir / "images")\n    yaml_path.parent.mkdir(parents=True, exist_ok=True)\n    with yaml_path.open("w", encoding="utf-8") as f:\n        yaml.safe_dump(content, f, sort_keys=False)\n    return yaml_path\n\n\ndef find_image_for_label(image_dir: Path, label_file: str | Path) -> Path | None:\n    stem = Path(label_file).stem\n    for ext in IMAGE_EXTENSIONS:\n        candidate = image_dir / f"{stem}{ext}"\n        if candidate.exists():\n            return candidate\n    return None\n\n\ndef count_labeled_images(label_dir: Path) -> dict[str, int]:\n    labeled = 0\n    healthy = 0\n    instances = 0\n    for label_path in sorted(label_dir.glob("*.txt")):\n        lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]\n        if lines:\n            labeled += 1\n            instances += len(lines)\n        else:\n            healthy += 1\n    return {"labeled_images": labeled, "healthy_images": healthy, "instances": instances}\n\n\ndef copy_split_by_label_state(src_dataset: Path, dst_dataset: Path, split: str, want_labeled: bool) -> int:\n    src_images = src_dataset / split / "images"\n    src_labels = src_dataset / split / "labels"\n    dst_images = dst_dataset / split / "images"\n    dst_labels = dst_dataset / split / "labels"\n    dst_images.mkdir(parents=True, exist_ok=True)\n    dst_labels.mkdir(parents=True, exist_ok=True)\n\n    copied = 0\n    for label_path in sorted(src_labels.glob("*.txt")):\n        lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]\n        if bool(lines) != want_labeled:\n            continue\n        image_path = find_image_for_label(src_images, label_path.name)\n        if image_path is None:\n            continue\n        image_dst = dst_images / image_path.name\n        label_dst = dst_labels / label_path.name\n        if not image_dst.exists():\n            shutil.copy2(image_path, image_dst)\n        if not label_dst.exists():\n            shutil.copy2(label_path, label_dst)\n        copied += 1\n    return copied\n\n\ndef make_state_eval_dataset(\n    src_dataset: Path,\n    source_data_yaml: Path,\n    output_root: Path,\n    tag: str,\n    state_name: str,\n    want_labeled: bool,\n) -> tuple[Path, Path, dict[str, int]]:\n    dst = output_root / "eval_datasets" / tag / f"dataset_{state_name}_eval"\n    yaml_path = dst / f"data_{state_name}.yaml"\n    if yaml_path.exists():\n        copied = {\n            "valid": count_labeled_images(dst / "valid" / "labels")["labeled_images" if want_labeled else "healthy_images"],\n            "test": count_labeled_images(dst / "test" / "labels")["labeled_images" if want_labeled else "healthy_images"],\n        }\n        return dst, yaml_path, copied\n\n    for sub in ("images", "labels"):\n        (dst / "train" / sub).mkdir(parents=True, exist_ok=True)\n    copied = {}\n    for split in ("valid", "test"):\n        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)\n    yaml_path = write_data_yaml(source_data_yaml, dst, yaml_path)\n    return dst, yaml_path, copied\n\n\ndef metric_value(metrics: Any, dotted_path: str, default: float = float("nan")) -> float:\n    obj = metrics\n    for part in dotted_path.split("."):\n        if isinstance(obj, dict):\n            if part not in obj:\n                return default\n            obj = obj[part]\n        elif hasattr(obj, part):\n            obj = getattr(obj, part)\n        else:\n            return default\n    try:\n        return float(obj)\n    except Exception:\n        return default\n\n\ndef result_dict_value(metrics: Any, keys: tuple[str, ...], default: float = float("nan")) -> float:\n    results_dict = getattr(metrics, "results_dict", None)\n    if not isinstance(results_dict, dict):\n        return default\n    for key in keys:\n        if key in results_dict:\n            try:\n                return float(results_dict[key])\n            except Exception:\n                return default\n    return default\n\n\ndef extract_val_metrics(metrics: Any, prefix: str) -> dict[str, float]:\n    speed = getattr(metrics, "speed", {}) or {}\n    preprocess_ms = float(speed.get("preprocess", float("nan"))) if isinstance(speed, dict) else float("nan")\n    inference_ms = float(speed.get("inference", float("nan"))) if isinstance(speed, dict) else float("nan")\n    postprocess_ms = float(speed.get("postprocess", float("nan"))) if isinstance(speed, dict) else float("nan")\n    total_ms = sum(v for v in (preprocess_ms, inference_ms, postprocess_ms) if not math.isnan(v))\n    fps = 1000.0 / total_ms if total_ms > 0 else float("nan")\n    return {\n        f"{prefix}_box_precision": metric_value(metrics, "box.mp", result_dict_value(metrics, ("metrics/precision(B)",))),\n        f"{prefix}_box_recall": metric_value(metrics, "box.mr", result_dict_value(metrics, ("metrics/recall(B)",))),\n        f"{prefix}_box_map50": metric_value(metrics, "box.map50", result_dict_value(metrics, ("metrics/mAP50(B)",))),\n        f"{prefix}_box_map50_95": metric_value(metrics, "box.map", result_dict_value(metrics, ("metrics/mAP50-95(B)",))),\n        f"{prefix}_mask_precision": metric_value(metrics, "seg.mp", result_dict_value(metrics, ("metrics/precision(M)",))),\n        f"{prefix}_mask_recall": metric_value(metrics, "seg.mr", result_dict_value(metrics, ("metrics/recall(M)",))),\n        f"{prefix}_mask_map50": metric_value(metrics, "seg.map50", result_dict_value(metrics, ("metrics/mAP50(M)",))),\n        f"{prefix}_mask_map50_95": metric_value(metrics, "seg.map", result_dict_value(metrics, ("metrics/mAP50-95(M)",))),\n        f"{prefix}_preprocess_ms": preprocess_ms,\n        f"{prefix}_inference_ms": inference_ms,\n        f"{prefix}_postprocess_ms": postprocess_ms,\n        f"{prefix}_fps": fps,\n    }\n\n\ndef image_paths_in_dir(images_dir: Path) -> list[Path]:\n    image_paths: list[Path] = []\n    for ext in IMAGE_EXTENSIONS:\n        image_paths.extend(images_dir.glob(f"*{ext}"))\n    return sorted(image_paths)\n\n\ndef count_prediction_errors(\n    model: Any,\n    images_dir: Path,\n    labels_dir: Path,\n    imgsz: int = DEFAULT_IMGSZ,\n    conf: float = DEFAULT_CONF,\n    iou: float = DEFAULT_IOU,\n    augment: bool = False,\n) -> dict[str, Any]:\n    image_paths = image_paths_in_dir(images_dir)\n    if not image_paths:\n        return {\n            "images": 0,\n            "gt_total": 0,\n            "pred_box_total": 0,\n            "pred_mask_total": 0,\n            "box_count_mae": float("nan"),\n            "mask_count_mae": float("nan"),\n            "box_count_exact": float("nan"),\n            "mask_count_exact": float("nan"),\n            "disease_images": 0,\n            "disease_box_miss_images": 0,\n            "disease_mask_miss_images": 0,\n            "disease_box_miss_rate": float("nan"),\n            "disease_mask_miss_rate": float("nan"),\n        }\n\n    results = model.predict(\n        source=[str(p) for p in image_paths],\n        imgsz=imgsz,\n        conf=conf,\n        iou=iou,\n        augment=augment,\n        verbose=False,\n    )\n    box_errors: list[float] = []\n    mask_errors: list[float] = []\n    box_exact: list[float] = []\n    mask_exact: list[float] = []\n    gt_total = 0\n    pred_box_total = 0\n    pred_mask_total = 0\n    disease_images = 0\n    disease_box_miss_images = 0\n    disease_mask_miss_images = 0\n\n    for image_path, result in zip(image_paths, results):\n        label_path = labels_dir / f"{image_path.stem}.txt"\n        gt_count = 0\n        if label_path.exists():\n            gt_count = len([line for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()])\n        box_count = len(result.boxes) if result.boxes is not None else 0\n        mask_count = len(result.masks) if result.masks is not None else 0\n        denom = max(1, gt_count)\n        box_errors.append(abs(box_count - gt_count) / denom)\n        mask_errors.append(abs(mask_count - gt_count) / denom)\n        box_exact.append(float(box_count == gt_count))\n        mask_exact.append(float(mask_count == gt_count))\n        if gt_count > 0:\n            disease_images += 1\n            disease_box_miss_images += int(box_count == 0)\n            disease_mask_miss_images += int(mask_count == 0)\n        gt_total += gt_count\n        pred_box_total += box_count\n        pred_mask_total += mask_count\n\n    return {\n        "images": len(image_paths),\n        "gt_total": gt_total,\n        "pred_box_total": pred_box_total,\n        "pred_mask_total": pred_mask_total,\n        "box_count_mae": sum(box_errors) / len(box_errors),\n        "mask_count_mae": sum(mask_errors) / len(mask_errors),\n        "box_count_exact": sum(box_exact) / len(box_exact),\n        "mask_count_exact": sum(mask_exact) / len(mask_exact),\n        "disease_images": disease_images,\n        "disease_box_miss_images": disease_box_miss_images,\n        "disease_mask_miss_images": disease_mask_miss_images,\n        "disease_box_miss_rate": disease_box_miss_images / disease_images if disease_images else float("nan"),\n        "disease_mask_miss_rate": disease_mask_miss_images / disease_images if disease_images else float("nan"),\n    }\n\n\ndef healthy_false_positive_summary(\n    model: Any,\n    images_dir: Path,\n    imgsz: int = DEFAULT_IMGSZ,\n    conf: float = DEFAULT_CONF,\n    iou: float = DEFAULT_IOU,\n    augment: bool = False,\n) -> dict[str, Any]:\n    image_paths = image_paths_in_dir(images_dir)\n    if not image_paths:\n        return {\n            "healthy_images": 0,\n            "healthy_images_with_box_fp": 0,\n            "healthy_images_with_mask_fp": 0,\n            "healthy_box_fp_rate": float("nan"),\n            "healthy_mask_fp_rate": float("nan"),\n            "healthy_fp_boxes_total": 0,\n            "healthy_fp_masks_total": 0,\n            "healthy_fp_boxes_per_image": float("nan"),\n            "healthy_fp_masks_per_image": float("nan"),\n            "healthy_avg_fp_confidence": float("nan"),\n        }\n\n    results = model.predict(\n        source=[str(p) for p in image_paths],\n        imgsz=imgsz,\n        conf=conf,\n        iou=iou,\n        augment=augment,\n        verbose=False,\n    )\n    images_with_box_fp = 0\n    images_with_mask_fp = 0\n    box_total = 0\n    mask_total = 0\n    confidences: list[float] = []\n\n    for result in results:\n        box_count = len(result.boxes) if result.boxes is not None else 0\n        mask_count = len(result.masks) if result.masks is not None else 0\n        if box_count > 0:\n            images_with_box_fp += 1\n            try:\n                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])\n            except Exception:\n                pass\n        if mask_count > 0:\n            images_with_mask_fp += 1\n        box_total += box_count\n        mask_total += mask_count\n\n    n = len(image_paths)\n    return {\n        "healthy_images": n,\n        "healthy_images_with_box_fp": images_with_box_fp,\n        "healthy_images_with_mask_fp": images_with_mask_fp,\n        "healthy_box_fp_rate": images_with_box_fp / n,\n        "healthy_mask_fp_rate": images_with_mask_fp / n,\n        "healthy_fp_boxes_total": box_total,\n        "healthy_fp_masks_total": mask_total,\n        "healthy_fp_boxes_per_image": box_total / n,\n        "healthy_fp_masks_per_image": mask_total / n,\n        "healthy_avg_fp_confidence": sum(confidences) / len(confidences) if confidences else 0.0,\n    }\n\n\ndef healthy_aware_score(labeled_map50: float, count_summary: dict[str, Any], healthy_fp_summary: dict[str, Any]) -> float:\n    return (\n        float(labeled_map50)\n        - COUNT_PENALTY_WEIGHT * float(count_summary["mask_count_mae"])\n        - DISEASE_MISS_PENALTY_WEIGHT * float(count_summary["disease_box_miss_rate"])\n        - HEALTHY_FP_PENALTY_WEIGHT * float(healthy_fp_summary["healthy_mask_fp_rate"])\n    )\n\n\ndef params_count(model: Any) -> int | None:\n    try:\n        return int(sum(p.numel() for p in model.model.parameters()))\n    except Exception:\n        return None\n\n\ndef evaluate_checkpoint(\n    candidate: dict[str, Any],\n    checkpoint: Path,\n    dataset_dir: Path,\n    data_yaml: Path,\n    output_root: Path,\n    imgsz: int = DEFAULT_IMGSZ,\n    conf: float = DEFAULT_CONF,\n    iou: float = DEFAULT_IOU,\n    augment: bool = False,\n    run_val: bool = False,\n    plots: bool = False,\n) -> dict[str, Any]:\n    from ultralytics import YOLO\n\n    validate_split_exists(dataset_dir)\n    if not data_yaml.exists():\n        raise FileNotFoundError(f"Data YAML not found: {data_yaml}")\n    if not checkpoint.exists():\n        raise FileNotFoundError(f"Checkpoint not found: {checkpoint}")\n\n    register_all_custom_attention_modules()\n    model = YOLO(str(checkpoint))\n    tag = f"{candidate[\'key\']}_{\'tta\' if augment else \'plain\'}_conf{conf:.2f}_iou{iou:.2f}".replace(".", "p")\n    labeled_dir, labeled_yaml, labeled_copied = make_state_eval_dataset(\n        dataset_dir, data_yaml, output_root, tag, "labeled_only", want_labeled=True\n    )\n    healthy_dir, healthy_yaml, healthy_copied = make_state_eval_dataset(\n        dataset_dir, data_yaml, output_root, tag, "healthy_only", want_labeled=False\n    )\n\n    start = time.time()\n    row: dict[str, Any] = {\n        "module_key": candidate["key"],\n        "display_name": candidate.get("display_name", candidate["key"]),\n        "family": candidate.get("family"),\n        "augmentation_policy": candidate.get("augmentation"),\n        "checkpoint": str(checkpoint),\n        "dataset_dir": str(dataset_dir),\n        "data_yaml": str(data_yaml),\n        "imgsz": imgsz,\n        "conf": conf,\n        "iou": iou,\n        "tta_augment": augment,\n        "params": params_count(model),\n        "labeled_eval_valid_images": labeled_copied.get("valid"),\n        "labeled_eval_test_images": labeled_copied.get("test"),\n        "healthy_eval_valid_images": healthy_copied.get("valid"),\n        "healthy_eval_test_images": healthy_copied.get("test"),\n    }\n    if run_val:\n        full_val_metrics = model.val(\n            data=str(data_yaml), split="val", imgsz=imgsz, conf=conf, iou=iou, augment=augment, plots=False, verbose=False\n        )\n        row.update(extract_val_metrics(full_val_metrics, "val"))\n\n    full_test_metrics = model.val(\n        data=str(data_yaml), split="test", imgsz=imgsz, conf=conf, iou=iou, augment=augment, plots=plots, verbose=False\n    )\n    labeled_test_metrics = model.val(\n        data=str(labeled_yaml), split="test", imgsz=imgsz, conf=conf, iou=iou, augment=augment, plots=False, verbose=False\n    )\n    eval_time_sec = time.time() - start\n    row.update(extract_val_metrics(full_test_metrics, "full_test"))\n    row.update(extract_val_metrics(labeled_test_metrics, "labeled_test"))\n\n    labeled_count = count_prediction_errors(\n        model, labeled_dir / "test" / "images", labeled_dir / "test" / "labels", imgsz=imgsz, conf=conf, iou=iou, augment=augment\n    )\n    full_count = count_prediction_errors(\n        model, dataset_dir / "test" / "images", dataset_dir / "test" / "labels", imgsz=imgsz, conf=conf, iou=iou, augment=augment\n    )\n    healthy_fp = healthy_false_positive_summary(\n        model, healthy_dir / "test" / "images", imgsz=imgsz, conf=conf, iou=iou, augment=augment\n    )\n    for key, value in labeled_count.items():\n        row[f"labeled_test_{key}"] = value\n    for key, value in full_count.items():\n        row[f"full_test_count_{key}"] = value\n    for key, value in healthy_fp.items():\n        row[f"healthy_test_{key}"] = value\n    row["healthy_aware_score"] = healthy_aware_score(row["labeled_test_mask_map50"], labeled_count, healthy_fp)\n    row["eval_time_sec"] = round(eval_time_sec, 4)\n    return row\n\n\ndef append_csv_row(path: Path, row: dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    existing_fields: list[str] = []\n    if path.exists():\n        with path.open("r", encoding="utf-8", newline="") as f:\n            reader = csv.reader(f)\n            existing_fields = next(reader, [])\n    fieldnames = list(existing_fields)\n    for key in row:\n        if key not in fieldnames:\n            fieldnames.append(key)\n\n    rows: list[dict[str, Any]] = []\n    if path.exists() and existing_fields:\n        with path.open("r", encoding="utf-8", newline="") as f:\n            rows = list(csv.DictReader(f))\n    rows.append(row)\n    with path.open("w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef existing_eval_keys(path: Path) -> set[tuple[str, float, float, bool]]:\n    if not path.exists():\n        return set()\n    keys: set[tuple[str, float, float, bool]] = set()\n    with path.open("r", encoding="utf-8", newline="") as f:\n        for row in csv.DictReader(f):\n            try:\n                keys.add(\n                    (\n                        row["module_key"],\n                        round(float(row["conf"]), 4),\n                        round(float(row["iou"]), 4),\n                        str(row.get("tta_augment", "")).lower() in {"true", "1", "yes"},\n                    )\n                )\n            except Exception:\n                continue\n    return keys\n\n\ndef disable_ultralytics_albumentations() -> None:\n    try:\n        import ultralytics.data.augment as yolo_augment\n    except Exception as exc:  # noqa: BLE001\n        print(f"Could not patch Ultralytics Albumentations hook: {exc}")\n        return\n\n    class NoOpAlbumentations:\n        contains_spatial = False\n\n        def __init__(self, *args: Any, **kwargs: Any) -> None:\n            self.transform = None\n\n        def __call__(self, labels: Any) -> Any:\n            return labels\n\n    yolo_augment.Albumentations = NoOpAlbumentations\n    print("Ultralytics Albumentations hook disabled for matched strong baseline.")\n\n\ndef sanity_check_model_yaml(model_yaml: str | Path, imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> dict[str, Any]:\n    """Instantiate a model YAML and run one dummy forward pass."""\n\n    from ultralytics import YOLO\n\n    register_all_custom_attention_modules()\n    model_path = resolve_existing_path(model_yaml)\n    if model_path is None:\n        raise FileNotFoundError(f"Model YAML not found: {model_yaml}")\n    yolo = YOLO(str(model_path))\n    try:\n        import torch\n\n        net = yolo.model.to(device).eval()\n        x = torch.zeros(1, 3, imgsz, imgsz, device=device)\n        with torch.no_grad():\n            output = net(x)\n        shapes = _shape_summary(output)\n    except Exception as exc:  # noqa: BLE001\n        raise RuntimeError(f"Dummy forward failed for {model_path}: {exc}") from exc\n\n    return {\n        "model_yaml": str(model_path),\n        "imgsz": imgsz,\n        "device": device,\n        "status": "ok",\n        "output_shapes": shapes,\n    }\n\n\ndef _shape_summary(value: Any) -> Any:\n    if hasattr(value, "shape"):\n        return tuple(int(v) for v in value.shape)\n    if isinstance(value, (list, tuple)):\n        return [_shape_summary(item) for item in value]\n    if isinstance(value, dict):\n        return {key: _shape_summary(item) for key, item in value.items()}\n    return type(value).__name__\n\n\ndef sanity_check_candidate_yamls(module_keys: list[str] | None = None, imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> list[dict[str, Any]]:\n    rows = []\n    for candidate in selected_candidates(module_keys):\n        model_yaml = candidate.get("model_yaml")\n        if not model_yaml or str(model_yaml).endswith(".pt"):\n            continue\n        row = {"module_key": candidate["key"], "display_name": candidate.get("display_name"), "model_yaml": str(model_yaml)}\n        try:\n            row.update(sanity_check_model_yaml(model_yaml, imgsz=imgsz, device=device))\n        except Exception as exc:  # noqa: BLE001\n            row.update({"status": "error", "error_type": exc.__class__.__name__, "error": str(exc)})\n        rows.append(row)\n    return rows\n', 'B': '"""Training-protocol helpers for custom-attention Group B.\n\nGroup B changes training protocol rather than architecture:\n- B1: warmup + cosine LR + shorter patience.\n- B2: runtime label smoothing for classification targets.\n- B3: matched underwater/strong augmentation.\n- B4: checkpoint averaging / SWA after training.\n\nThe evaluation path delegates to Group A utilities so metric definitions stay\nidentical across calibration and retraining experiments.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport importlib.util\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n\nDEFAULT_IMGSZ = 640\nDEFAULT_BATCH = 16\nDEFAULT_SEED = 42\nDEFAULT_EPOCHS = 100\nDEFAULT_PATIENCE = 30\n\nTOP_PROTOCOL_MODULE_KEYS = [\n    "baseline_clean",\n    "triplet_attention_segment_head",\n    "cote_gate",\n    "lpsc_gate",\n    "cesa_lite_segment_head",\n    "sge_eca_head_gate",\n]\n\nCLEAN_LIGHT_AUG_TRAIN_ARGS: dict[str, Any] = {\n    "auto_augment": None,\n    "erasing": 0.0,\n    "mosaic": 0.0,\n    "mixup": 0.0,\n    "cutmix": 0.0,\n    "copy_paste": 0.0,\n    "fliplr": 0.5,\n    "flipud": 0.0,\n    "hsv_h": 0.01,\n    "hsv_s": 0.35,\n    "hsv_v": 0.20,\n    "degrees": 0.0,\n    "translate": 0.05,\n    "scale": 0.20,\n    "shear": 0.0,\n    "perspective": 0.0,\n    "multi_scale": 0.0,\n    "bgr": 0.0,\n}\n\nSTABLE_TRAIN_CONTROL: dict[str, Any] = {\n    "epochs": 150,\n    "patience": 20,\n    "cos_lr": True,\n    "warmup_epochs": 8,\n    "lr0": 0.01,\n    "lrf": 0.001,\n}\n\nUNDERWATER_AUG_ARGS: dict[str, Any] = {\n    "auto_augment": None,\n    "mosaic": 0.0,\n    "mixup": 0.0,\n    "copy_paste": 0.0,\n    "cutmix": 0.0,\n    "fliplr": 0.5,\n    "flipud": 0.3,\n    "hsv_h": 0.05,\n    "hsv_s": 0.50,\n    "hsv_v": 0.40,\n    "scale": 0.50,\n    "translate": 0.10,\n    "degrees": 10.0,\n    "erasing": 0.10,\n    "shear": 0.0,\n    "perspective": 0.0,\n    "multi_scale": 0.0,\n    "bgr": 0.0,\n}\n\n\ndef attention_root() -> Path:\n    here = Path(__file__).resolve()\n    for parent in [*here.parents, Path.cwd(), Path.cwd() / "yolov11n_attention"]:\n        if (parent / "custom_attention").exists() and (parent / "yolov11n_custom_attention_NhomA").exists():\n            return parent\n    return here.parents[2]\n\n\ndef load_group_a_utils():\n    """Load Group A utilities under a private module name to avoid _shared name clashes."""\n\n    root = attention_root()\n    path = root / "yolov11n_custom_attention_NhomA" / "_shared" / "nhomA_eval_utils.py"\n    if not path.exists():\n        raise FileNotFoundError(f"Group A helper not found: {path}")\n    module_name = "nhomA_eval_utils_for_nhomB"\n    if module_name in sys.modules:\n        return sys.modules[module_name]\n    spec = importlib.util.spec_from_file_location(module_name, path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Could not load Group A helper from {path}")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[module_name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef ensure_project_import_paths() -> Path:\n    a = load_group_a_utils()\n    return a.ensure_project_import_paths()\n\n\ndef register_all_custom_attention_modules() -> None:\n    a = load_group_a_utils()\n    a.register_all_custom_attention_modules()\n\n\ndef selected_candidates(module_keys: list[str] | None = None) -> list[dict[str, Any]]:\n    a = load_group_a_utils()\n    return a.selected_candidates(module_keys or TOP_PROTOCOL_MODULE_KEYS)\n\n\ndef candidate_by_key(key: str) -> dict[str, Any]:\n    a = load_group_a_utils()\n    return a.candidate_by_key(key)\n\n\ndef resolve_existing_path(path_value: str | Path | None) -> Path | None:\n    a = load_group_a_utils()\n    return a.resolve_existing_path(path_value)\n\n\ndef validate_split_exists(dataset_dir: Path) -> None:\n    a = load_group_a_utils()\n    a.validate_split_exists(dataset_dir)\n\n\ndef write_data_yaml(source_data_yaml: Path, dataset_dir: Path, yaml_path: Path) -> Path:\n    a = load_group_a_utils()\n    return a.write_data_yaml(source_data_yaml, dataset_dir, yaml_path)\n\n\ndef copy_dataset_snapshot(source_dataset: Path, snapshot_dir: Path, refresh: bool = False) -> Path:\n    a = load_group_a_utils()\n    return a.copy_dataset_snapshot(source_dataset, snapshot_dir, refresh=refresh)\n\n\ndef append_csv_row(path: Path, row: dict[str, Any]) -> None:\n    a = load_group_a_utils()\n    a.append_csv_row(path, row)\n\n\ndef disable_ultralytics_albumentations() -> None:\n    a = load_group_a_utils()\n    a.disable_ultralytics_albumentations()\n\n\ndef evaluate_checkpoint(*args, **kwargs) -> dict[str, Any]:\n    a = load_group_a_utils()\n    return a.evaluate_checkpoint(*args, **kwargs)\n\n\ndef sanity_check_candidate_yamls(module_keys: list[str] | None = None, imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> list[dict[str, Any]]:\n    a = load_group_a_utils()\n    return a.sanity_check_candidate_yamls(module_keys or TOP_PROTOCOL_MODULE_KEYS, imgsz=imgsz, device=device)\n\n\ndef read_best_epoch_from_results(run_path: Path) -> dict[str, Any]:\n    results_csv = Path(run_path) / "results.csv"\n    if not results_csv.exists():\n        return {}\n    try:\n        import pandas as pd\n    except Exception:\n        return {"results_csv": str(results_csv)}\n    df = pd.read_csv(results_csv)\n    df.columns = [str(c).strip() for c in df.columns]\n    mask_col = "metrics/mAP50(M)"\n    if mask_col not in df.columns:\n        return {"epochs_ran": int(len(df)), "results_csv": str(results_csv)}\n    best_idx = df[mask_col].idxmax()\n    first = df.iloc[0]\n    best = df.iloc[best_idx]\n    last = df.iloc[-1]\n    return {\n        "epochs_ran": int(len(df)),\n        "best_epoch_by_mask_map50": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),\n        "first_train_seg_loss": float(first.get("train/seg_loss", float("nan"))),\n        "best_val_mask_map50": float(best.get(mask_col, float("nan"))),\n        "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),\n        "last_val_mask_map50": float(last.get(mask_col, float("nan"))),\n        "last_val_mask_map50_95": float(last.get("metrics/mAP50-95(M)", float("nan"))),\n        "last_train_seg_loss": float(last.get("train/seg_loss", float("nan"))),\n        "last_val_seg_loss": float(last.get("val/seg_loss", float("nan"))),\n        "seg_loss_gap_val_minus_train": float(\n            last.get("val/seg_loss", float("nan")) - last.get("train/seg_loss", float("nan"))\n        ),\n        "results_csv": str(results_csv),\n    }\n\n\ndef train_args_without_removed_keys(train_args: dict[str, Any]) -> dict[str, Any]:\n    """Avoid passing removed Ultralytics keys that silently become no-ops."""\n\n    removed = {"label_smoothing", "save_hybrid", "crop_fraction"}\n    return {k: v for k, v in dict(train_args).items() if k not in removed}\n\n\ndef enable_runtime_label_smoothing(eps: float = 0.05) -> None:\n    """Patch v8DetectionLoss target scores for true runtime label smoothing.\n\n    Current vendored Ultralytics removes the train arg `label_smoothing`, so this\n    patch applies smoothing directly to BCE classification targets. It is\n    process-local and does not edit Ultralytics files on disk.\n    """\n\n    import torch\n    import ultralytics.utils.loss as loss_mod\n    from ultralytics.utils.loss import make_anchors, xywh2xyxy\n\n    cls = loss_mod.v8DetectionLoss\n    if getattr(cls, "_nhomb_label_smoothing_patched", False):\n        cls._nhomb_label_smoothing_eps = float(eps)\n        return\n\n    original = cls.get_assigned_targets_and_loss\n\n    def patched_get_assigned_targets_and_loss(self, preds: dict[str, torch.Tensor], batch: dict[str, Any]) -> tuple:\n        loss = torch.zeros(3, device=self.device)\n        pred_distri, pred_scores = (\n            preds["boxes"].permute(0, 2, 1).contiguous(),\n            preds["scores"].permute(0, 2, 1).contiguous(),\n        )\n        anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5)\n\n        dtype = pred_scores.dtype\n        batch_size = pred_scores.shape[0]\n        imgsz = torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]\n\n        targets = torch.cat((batch["batch_idx"].view(-1, 1), batch["cls"].view(-1, 1), batch["bboxes"]), 1)\n        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])\n        gt_labels, gt_bboxes = targets.split((1, 4), 2)\n        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)\n\n        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)\n\n        _, target_bboxes, target_scores, fg_mask, target_gt_idx = self.assigner(\n            pred_scores.detach().sigmoid(),\n            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),\n            anchor_points * stride_tensor,\n            gt_labels,\n            gt_bboxes,\n            mask_gt,\n        )\n\n        target_scores_sum = max(target_scores.sum(), 1)\n\n        eps_value = float(getattr(cls, "_nhomb_label_smoothing_eps", eps))\n        if eps_value > 0:\n            smooth_neg = eps_value / max(1, self.nc)\n            cls_targets = target_scores.to(dtype) * (1.0 - eps_value) + smooth_neg\n        else:\n            cls_targets = target_scores.to(dtype)\n\n        bce_loss = self.bce(pred_scores, cls_targets)\n        if self.class_weights is not None:\n            bce_loss *= self.class_weights\n        loss[1] = bce_loss.sum() / target_scores_sum\n\n        if fg_mask.sum():\n            loss[0], loss[2] = self.bbox_loss(\n                pred_distri,\n                pred_bboxes,\n                anchor_points,\n                target_bboxes / stride_tensor,\n                target_scores,\n                target_scores_sum,\n                fg_mask,\n                imgsz,\n                stride_tensor,\n            )\n\n        loss[0] *= self.hyp.box\n        loss[1] *= self.hyp.cls\n        loss[2] *= self.hyp.dfl\n        return (fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor), loss, loss.detach()\n\n    patched_get_assigned_targets_and_loss.__name__ = original.__name__\n    patched_get_assigned_targets_and_loss.__doc__ = original.__doc__\n    cls.get_assigned_targets_and_loss = patched_get_assigned_targets_and_loss\n    cls._nhomb_label_smoothing_patched = True\n    cls._nhomb_label_smoothing_eps = float(eps)\n\n\ndef candidate_model_spec(candidate: dict[str, Any]) -> str:\n    value = candidate.get("model_yaml")\n    if not value or str(value).endswith(".pt"):\n        return str(value or "yolo11n-seg.pt")\n    resolved = resolve_existing_path(value)\n    if resolved is None:\n        raise FileNotFoundError(f"Model YAML not found for {candidate[\'key\']}: {value}")\n    return str(resolved)\n\n\ndef run_dir_for(runs_root: Path, experiment_key: str, module_key: str, seed: int) -> Path:\n    return Path(runs_root) / experiment_key / module_key / f"seed_{seed}"\n\n\ndef run_training_experiment(\n    candidate_key: str,\n    experiment_key: str,\n    base_data_dir: Path,\n    source_data_yaml: Path,\n    output_root: Path,\n    runs_root: Path,\n    train_args: dict[str, Any],\n    seed: int = DEFAULT_SEED,\n    imgsz: int = DEFAULT_IMGSZ,\n    epochs: int = DEFAULT_EPOCHS,\n    batch: int = DEFAULT_BATCH,\n    patience: int = DEFAULT_PATIENCE,\n    smoke: bool = False,\n    save_period: int = -1,\n    pretrained_weights: str = "yolo11n-seg.pt",\n    load_pretrained_weights: bool = True,\n    label_smoothing_eps: float | None = None,\n    evaluate_after: bool = True,\n    eval_conf: float = 0.25,\n    eval_iou: float = 0.70,\n) -> dict[str, Any]:\n    from ultralytics import YOLO\n\n    ensure_project_import_paths()\n    register_all_custom_attention_modules()\n    if label_smoothing_eps is not None:\n        enable_runtime_label_smoothing(label_smoothing_eps)\n\n    candidate = candidate_by_key(candidate_key)\n    model_spec = candidate_model_spec(candidate)\n    exp_output = Path(output_root) / experiment_key\n    dataset_dir = exp_output / "datasets" / candidate_key / f"seed_{seed}" / "dataset"\n    dataset_dir = copy_dataset_snapshot(Path(base_data_dir), dataset_dir, refresh=False)\n    data_yaml = write_data_yaml(Path(source_data_yaml), dataset_dir, dataset_dir / "data.yaml")\n\n    run_dir = run_dir_for(Path(runs_root), experiment_key, candidate_key, seed)\n    best_path = run_dir / "weights" / "best.pt"\n    train_csv = exp_output / "reports" / f"{experiment_key}_train_runs.csv"\n    eval_csv = exp_output / "reports" / f"{experiment_key}_eval_results.csv"\n\n    train_row: dict[str, Any] = {\n        "experiment_key": experiment_key,\n        "module_key": candidate_key,\n        "display_name": candidate.get("display_name", candidate_key),\n        "seed": seed,\n        "model_spec": model_spec,\n        "dataset_dir": str(dataset_dir),\n        "data_yaml": str(data_yaml),\n        "run_dir": str(run_dir),\n        "best_checkpoint": str(best_path),\n        "smoke": smoke,\n        "label_smoothing_eps": label_smoothing_eps,\n    }\n\n    if best_path.exists():\n        train_row.update({"status": "skipped_existing", "reason": "best checkpoint already exists"})\n    else:\n        disable_ultralytics_albumentations()\n        yolo = YOLO(model_spec)\n        if load_pretrained_weights and not str(model_spec).endswith(".pt"):\n            yolo = yolo.load(str(pretrained_weights))\n\n        actual_imgsz = 320 if smoke else int(imgsz)\n        actual_epochs = 1 if smoke else int(epochs)\n        actual_batch = 8 if smoke else int(batch)\n        actual_patience = 1 if smoke else int(patience)\n        clean_train_args = train_args_without_removed_keys(train_args)\n\n        start = time.time()\n        yolo.train(\n            data=str(data_yaml),\n            task="segment",\n            imgsz=actual_imgsz,\n            epochs=actual_epochs,\n            batch=actual_batch,\n            patience=actual_patience,\n            seed=seed,\n            project=str(run_dir.parent),\n            name=run_dir.name,\n            exist_ok=False,\n            pretrained=True,\n            plots=not smoke,\n            verbose=True,\n            save_period=save_period,\n            **clean_train_args,\n        )\n        train_row.update(\n            {\n                "status": "trained",\n                "train_time_min": round((time.time() - start) / 60.0, 4),\n                "imgsz": actual_imgsz,\n                "epochs": actual_epochs,\n                "batch": actual_batch,\n                "patience": actual_patience,\n                "save_period": save_period,\n                "train_args_json": json.dumps(clean_train_args, sort_keys=True),\n            }\n        )\n\n    train_row.update(read_best_epoch_from_results(run_dir))\n    append_csv_row(train_csv, train_row)\n\n    if evaluate_after and best_path.exists():\n        eval_row = evaluate_checkpoint(\n            candidate={**candidate, "augmentation": experiment_key},\n            checkpoint=best_path,\n            dataset_dir=dataset_dir,\n            data_yaml=data_yaml,\n            output_root=exp_output,\n            imgsz=320 if smoke else int(imgsz),\n            conf=eval_conf,\n            iou=eval_iou,\n            augment=False,\n            run_val=True,\n            plots=False,\n        )\n        eval_row.update({"experiment_key": experiment_key, "seed": seed, "protocol_train_status": train_row.get("status")})\n        append_csv_row(eval_csv, eval_row)\n        train_row["eval_csv"] = str(eval_csv)\n\n    train_row["train_csv"] = str(train_csv)\n    return train_row\n\n\ndef checkpoint_epoch_number(path: Path) -> int:\n    match = re.search(r"epoch(\\d+)\\.pt$", path.name)\n    return int(match.group(1)) if match else -1\n\n\ndef swa_source_checkpoints(run_dir: Path, max_checkpoints: int = 5, include_last: bool = True) -> list[Path]:\n    weights_dir = Path(run_dir) / "weights"\n    epoch_paths = sorted(weights_dir.glob("epoch*.pt"), key=checkpoint_epoch_number)\n    selected = epoch_paths[-max_checkpoints:]\n    last = weights_dir / "last.pt"\n    if include_last and last.exists() and last not in selected:\n        selected.append(last)\n    return [path for path in selected if path.exists()]\n\n\ndef _torch_load(path: Path) -> dict[str, Any]:\n    import torch\n\n    try:\n        return torch.load(path, map_location="cpu", weights_only=False)\n    except TypeError:\n        return torch.load(path, map_location="cpu")\n\n\ndef average_yolo_checkpoints(checkpoints: list[Path], output_path: Path) -> Path:\n    """Average floating parameters from multiple YOLO checkpoints."""\n\n    import torch\n\n    if len(checkpoints) < 2:\n        raise ValueError("SWA needs at least two checkpoints to average.")\n    register_all_custom_attention_modules()\n\n    ckpts = [_torch_load(Path(path)) for path in checkpoints]\n    model_key = "ema" if ckpts[0].get("ema") is not None else "model"\n    models = [ckpt.get(model_key) or ckpt.get("model") for ckpt in ckpts]\n    if any(model is None for model in models):\n        raise ValueError("At least one checkpoint does not contain a YOLO model.")\n\n    states = [model.float().state_dict() for model in models]\n    first_state = states[0]\n    avg_state = {}\n    for key, tensor in first_state.items():\n        if not torch.is_tensor(tensor):\n            avg_state[key] = tensor\n            continue\n        if tensor.dtype.is_floating_point and all(key in state and torch.is_tensor(state[key]) for state in states[1:]):\n            stacked = torch.stack([state[key].float() for state in states], dim=0)\n            avg_state[key] = stacked.mean(dim=0).to(dtype=tensor.dtype)\n        else:\n            avg_state[key] = tensor\n\n    averaged_model = copy.deepcopy(models[0]).float()\n    averaged_model.load_state_dict(avg_state, strict=False)\n    out_ckpt = dict(ckpts[0])\n    out_ckpt["model"] = averaged_model\n    out_ckpt["ema"] = averaged_model\n    out_ckpt["train_args"] = dict(out_ckpt.get("train_args", {}), nhomb_swa_sources=[str(p) for p in checkpoints])\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(out_ckpt, output_path)\n    return output_path\n\n\ndef write_json(path: Path, payload: dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as f:\n        json.dump(payload, f, indent=2, sort_keys=True)\n\n', 'C': '"""Mask-head and boundary-quality helpers for custom-attention Group C.\n\nGroup C is intentionally ablation focused:\n- C1: increase Segment mask coefficient/prototype count from nm=32 to nm=64.\n- C2: add a runtime boundary-aware mask objective.\n- C3: replace nearest-neighbor neck/head upsampling with bilinear upsampling.\n\nThe helper reuses Group A for custom module registration and evaluation, and\nGroup B for training protocol defaults. It does not edit vendored Ultralytics\nfiles or existing experiment notebooks.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport importlib.util\nimport json\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n\nDEFAULT_IMGSZ = 640\nDEFAULT_BATCH = 16\nDEFAULT_SEED = 42\nDEFAULT_EPOCHS = 100\nDEFAULT_PATIENCE = 30\n\nMASK_HEAD_MODULE_KEYS = [\n    "triplet_attention_segment_head",\n    "cote_gate",\n    "cesa_lite_segment_head",\n    "sge_eca_head_gate",\n    "lpsc_gate",\n]\n\nBOUNDARY_LOSS_MODULE_KEYS = [\n    "triplet_attention_segment_head",\n    "cote_gate",\n    "cesa_lite_segment_head",\n]\n\nBILINEAR_UPSAMPLE_MODULE_KEYS = [\n    "baseline_clean",\n    "triplet_attention_segment_head",\n    "cote_gate",\n    "cesa_lite_segment_head",\n    "sge_eca_head_gate",\n]\n\n\ndef attention_root() -> Path:\n    here = Path(__file__).resolve()\n    for parent in [*here.parents, Path.cwd(), Path.cwd() / "yolov11n_attention"]:\n        if (parent / "custom_attention").exists() and (parent / "yolov11n_custom_attention_NhomA").exists():\n            return parent\n    return here.parents[2]\n\n\ndef _load_module_from_file(module_name: str, path: Path):\n    if module_name in sys.modules:\n        return sys.modules[module_name]\n    if not path.exists():\n        raise FileNotFoundError(f"Required helper not found: {path}")\n    spec = importlib.util.spec_from_file_location(module_name, path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Could not load helper from {path}")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[module_name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_group_a_utils():\n    root = attention_root()\n    path = root / "yolov11n_custom_attention_NhomA" / "_shared" / "nhomA_eval_utils.py"\n    return _load_module_from_file("nhomA_eval_utils_for_nhomC", path)\n\n\ndef load_group_b_utils():\n    root = attention_root()\n    path = root / "yolov11n_custom_attention_NhomB" / "_shared" / "nhomB_train_utils.py"\n    return _load_module_from_file("nhomB_train_utils_for_nhomC", path)\n\n\ndef ensure_project_import_paths() -> Path:\n    return load_group_a_utils().ensure_project_import_paths()\n\n\ndef register_all_custom_attention_modules() -> None:\n    load_group_a_utils().register_all_custom_attention_modules()\n\n\ndef candidate_by_key(key: str) -> dict[str, Any]:\n    return load_group_a_utils().candidate_by_key(key)\n\n\ndef selected_candidates(module_keys: list[str] | None = None) -> list[dict[str, Any]]:\n    return load_group_a_utils().selected_candidates(module_keys)\n\n\ndef resolve_existing_path(path_value: str | Path | None) -> Path | None:\n    return load_group_a_utils().resolve_existing_path(path_value)\n\n\ndef copy_dataset_snapshot(source_dataset: Path, snapshot_dir: Path, refresh: bool = False) -> Path:\n    return load_group_a_utils().copy_dataset_snapshot(source_dataset, snapshot_dir, refresh=refresh)\n\n\ndef write_data_yaml(source_data_yaml: Path, dataset_dir: Path, yaml_path: Path) -> Path:\n    return load_group_a_utils().write_data_yaml(source_data_yaml, dataset_dir, yaml_path)\n\n\ndef append_csv_row(path: Path, row: dict[str, Any]) -> None:\n    load_group_a_utils().append_csv_row(path, row)\n\n\ndef disable_ultralytics_albumentations() -> None:\n    load_group_a_utils().disable_ultralytics_albumentations()\n\n\ndef evaluate_checkpoint(*args, **kwargs) -> dict[str, Any]:\n    return load_group_a_utils().evaluate_checkpoint(*args, **kwargs)\n\n\ndef sanity_check_model_yaml(model_yaml: str | Path, imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> dict[str, Any]:\n    return load_group_a_utils().sanity_check_model_yaml(model_yaml, imgsz=imgsz, device=device)\n\n\ndef read_best_epoch_from_results(run_path: Path) -> dict[str, Any]:\n    return load_group_b_utils().read_best_epoch_from_results(run_path)\n\n\ndef train_args_without_removed_keys(train_args: dict[str, Any]) -> dict[str, Any]:\n    return load_group_b_utils().train_args_without_removed_keys(train_args)\n\n\ndef clean_light_train_args() -> dict[str, Any]:\n    return copy.deepcopy(load_group_b_utils().CLEAN_LIGHT_AUG_TRAIN_ARGS)\n\n\ndef stable_train_control() -> dict[str, Any]:\n    return copy.deepcopy(load_group_b_utils().STABLE_TRAIN_CONTROL)\n\n\ndef run_dir_for(runs_root: Path, experiment_key: str, module_key: str, seed: int) -> Path:\n    return load_group_b_utils().run_dir_for(Path(runs_root), experiment_key, module_key, seed)\n\n\ndef _module_name(module_value: Any) -> str:\n    if isinstance(module_value, str):\n        return module_value\n    return getattr(module_value, "__name__", str(module_value))\n\n\ndef _iter_model_layers(cfg: dict[str, Any]):\n    for section in ("backbone", "head"):\n        for index, layer in enumerate(cfg.get(section, [])):\n            if isinstance(layer, list) and len(layer) >= 4:\n                yield section, index, layer\n\n\ndef mutate_segment_nm(cfg: dict[str, Any], nm: int = 64) -> list[dict[str, Any]]:\n    """Change only the Segment nm argument while preserving npr."""\n\n    changes: list[dict[str, Any]] = []\n    for section, index, layer in _iter_model_layers(cfg):\n        if _module_name(layer[2]) != "Segment":\n            continue\n        args = layer[3]\n        if not isinstance(args, list) or len(args) < 2:\n            raise ValueError(f"Segment layer at {section}[{index}] does not have [nc, nm, ...] args: {layer}")\n        old_nm = args[1]\n        args[1] = int(nm)\n        changes.append(\n            {\n                "section": section,\n                "layer_index": index,\n                "old_nm": old_nm,\n                "new_nm": int(nm),\n                "segment_args_after": list(args),\n            }\n        )\n    if not changes:\n        raise ValueError("No Segment layer found while applying nm mutation.")\n    return changes\n\n\ndef mutate_upsample_mode(cfg: dict[str, Any], mode: str = "bilinear") -> list[dict[str, Any]]:\n    """Change neck/head nn.Upsample mode, leaving size/scale_factor unchanged."""\n\n    changes: list[dict[str, Any]] = []\n    for section, index, layer in _iter_model_layers(cfg):\n        if _module_name(layer[2]) != "nn.Upsample":\n            continue\n        args = layer[3]\n        if not isinstance(args, list) or len(args) < 3:\n            raise ValueError(f"nn.Upsample layer at {section}[{index}] does not have [size, scale, mode]: {layer}")\n        old_mode = args[2]\n        args[2] = str(mode)\n        changes.append(\n            {\n                "section": section,\n                "layer_index": index,\n                "old_mode": old_mode,\n                "new_mode": str(mode),\n                "upsample_args_after": list(args),\n            }\n        )\n    if not changes:\n        raise ValueError("No nn.Upsample layer found while applying upsample mutation.")\n    return changes\n\n\ndef source_yaml_for_candidate(candidate_key: str) -> Path:\n    if candidate_key == "baseline_clean":\n        baseline_yaml = (\n            attention_root()\n            / "yolov11n_custom_attention_NhomC"\n            / "_shared"\n            / "yolo11n_seg_resolved_baseline.yaml"\n        )\n        if baseline_yaml.exists():\n            return baseline_yaml\n        raise FileNotFoundError(f"Resolved baseline YAML not found: {baseline_yaml}")\n\n    candidate = candidate_by_key(candidate_key)\n    model_yaml = candidate.get("model_yaml")\n    if not model_yaml or str(model_yaml).endswith(".pt"):\n        raise ValueError(\n            f"Candidate {candidate_key!r} does not expose a source YAML. "\n            "Pass a custom YAML candidate or add a baseline YAML before generating mask-head variants."\n        )\n    resolved = resolve_existing_path(model_yaml)\n    if resolved is None:\n        raise FileNotFoundError(f"Model YAML not found for {candidate_key}: {model_yaml}")\n    return resolved\n\n\ndef create_variant_yaml(\n    candidate_key: str,\n    variant_key: str,\n    output_dir: Path,\n    nm: int | None = None,\n    upsample_mode: str | None = None,\n) -> dict[str, Any]:\n    """Create a Group C YAML variant for a candidate module."""\n\n    source_yaml = source_yaml_for_candidate(candidate_key)\n    with source_yaml.open("r", encoding="utf-8") as f:\n        cfg = yaml.safe_load(f)\n    if not isinstance(cfg, dict):\n        raise ValueError(f"YAML did not parse to a mapping: {source_yaml}")\n\n    segment_changes: list[dict[str, Any]] = []\n    upsample_changes: list[dict[str, Any]] = []\n    if nm is not None:\n        segment_changes = mutate_segment_nm(cfg, nm=int(nm))\n    if upsample_mode is not None:\n        upsample_changes = mutate_upsample_mode(cfg, mode=str(upsample_mode))\n\n    out_dir = Path(output_dir) / "generated_yamls" / candidate_key\n    out_dir.mkdir(parents=True, exist_ok=True)\n    out_yaml = out_dir / f"{candidate_key}_{variant_key}.yaml"\n    with out_yaml.open("w", encoding="utf-8") as f:\n        yaml.safe_dump(cfg, f, sort_keys=False)\n\n    metadata = {\n        "candidate_key": candidate_key,\n        "variant_key": variant_key,\n        "source_yaml": str(source_yaml),\n        "variant_yaml": str(out_yaml),\n        "segment_nm_changes": len(segment_changes),\n        "upsample_mode_changes": len(upsample_changes),\n        "segment_change_details": json.dumps(segment_changes, sort_keys=True),\n        "upsample_change_details": json.dumps(upsample_changes, sort_keys=True),\n    }\n    return metadata\n\n\ndef create_variant_yamls(\n    module_keys: list[str],\n    variant_key: str,\n    output_dir: Path,\n    nm: int | None = None,\n    upsample_mode: str | None = None,\n) -> list[dict[str, Any]]:\n    rows = []\n    for module_key in module_keys:\n        rows.append(create_variant_yaml(module_key, variant_key, output_dir, nm=nm, upsample_mode=upsample_mode))\n    return rows\n\n\ndef sanity_check_variant_yamls(rows: list[dict[str, Any]], imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> list[dict[str, Any]]:\n    checked = []\n    for row in rows:\n        item = dict(row)\n        try:\n            item.update(sanity_check_model_yaml(item["variant_yaml"], imgsz=imgsz, device=device))\n        except Exception as exc:  # noqa: BLE001\n            item.update({"status": "error", "error_type": exc.__class__.__name__, "error": str(exc)})\n        checked.append(item)\n    return checked\n\n\ndef _sobel_boundary_map(mask):\n    import torch\n    import torch.nn.functional as F\n\n    if mask.ndim == 2:\n        mask = mask.unsqueeze(0)\n    if mask.numel() == 0:\n        return mask\n    x = mask.float().unsqueeze(1)\n    kx = torch.tensor(\n        [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],\n        device=x.device,\n        dtype=x.dtype,\n    ).view(1, 1, 3, 3)\n    ky = torch.tensor(\n        [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],\n        device=x.device,\n        dtype=x.dtype,\n    ).view(1, 1, 3, 3)\n    gx = F.conv2d(x, kx, padding=1)\n    gy = F.conv2d(x, ky, padding=1)\n    mag = torch.sqrt(gx.square() + gy.square() + 1e-6).squeeze(1)\n    max_per_mask = mag.amax(dim=(-2, -1), keepdim=True).clamp_min(1e-6)\n    return (mag / max_per_mask).clamp(0.0, 1.0)\n\n\ndef enable_boundary_aware_mask_loss(boundary_weight: float = 0.10, edge_mode: str = "sobel") -> None:\n    """Patch v8SegmentationLoss.single_mask_loss with a boundary objective.\n\n    The patch is process-local. It keeps the original BCE mask loss and adds a\n    cropped, area-normalized L1 penalty between Sobel boundary maps of predicted\n    probabilities and ground-truth masks.\n    """\n\n    if edge_mode != "sobel":\n        raise ValueError("Only edge_mode=\'sobel\' is currently supported.")\n\n    import torch\n    import torch.nn.functional as F\n    import ultralytics.utils.loss as loss_mod\n\n    cls = loss_mod.v8SegmentationLoss\n    if not hasattr(cls, "_nhomc_original_single_mask_loss"):\n        cls._nhomc_original_single_mask_loss = cls.single_mask_loss\n\n    weight = float(boundary_weight)\n\n    def boundary_single_mask_loss(gt_mask, pred, proto, xyxy, area):\n        pred_mask = torch.einsum("in,nhw->ihw", pred, proto)\n        base_loss = F.binary_cross_entropy_with_logits(pred_mask, gt_mask, reduction="none")\n\n        pred_edge = _sobel_boundary_map(pred_mask.sigmoid())\n        gt_edge = _sobel_boundary_map(gt_mask.float())\n        boundary_loss = F.l1_loss(pred_edge, gt_edge, reduction="none")\n\n        total_loss = base_loss + weight * boundary_loss\n        safe_area = area.clamp_min(1e-6)\n        return (loss_mod.crop_mask(total_loss, xyxy).mean(dim=(1, 2)) / safe_area).sum()\n\n    cls.single_mask_loss = staticmethod(boundary_single_mask_loss)\n    cls._nhomc_boundary_loss_patched = True\n    cls._nhomc_boundary_loss_weight = weight\n    cls._nhomc_boundary_loss_edge_mode = edge_mode\n\n\ndef disable_boundary_aware_mask_loss() -> None:\n    import ultralytics.utils.loss as loss_mod\n\n    cls = loss_mod.v8SegmentationLoss\n    original = getattr(cls, "_nhomc_original_single_mask_loss", None)\n    if original is not None:\n        cls.single_mask_loss = staticmethod(original)\n    cls._nhomc_boundary_loss_patched = False\n\n\ndef boundary_loss_patch_status() -> dict[str, Any]:\n    import ultralytics.utils.loss as loss_mod\n\n    cls = loss_mod.v8SegmentationLoss\n    return {\n        "patched": bool(getattr(cls, "_nhomc_boundary_loss_patched", False)),\n        "boundary_weight": getattr(cls, "_nhomc_boundary_loss_weight", None),\n        "edge_mode": getattr(cls, "_nhomc_boundary_loss_edge_mode", None),\n    }\n\n\ndef run_mask_head_variant_experiment(\n    candidate_key: str,\n    experiment_key: str,\n    variant_key: str,\n    base_data_dir: Path,\n    source_data_yaml: Path,\n    output_root: Path,\n    runs_root: Path,\n    train_args: dict[str, Any],\n    seed: int = DEFAULT_SEED,\n    imgsz: int = DEFAULT_IMGSZ,\n    epochs: int = DEFAULT_EPOCHS,\n    batch: int = DEFAULT_BATCH,\n    patience: int = DEFAULT_PATIENCE,\n    smoke: bool = False,\n    save_period: int = -1,\n    pretrained_weights: str = "yolo11n-seg.pt",\n    load_pretrained_weights: bool = True,\n    nm: int | None = None,\n    upsample_mode: str | None = None,\n    boundary_loss_weight: float | None = None,\n    evaluate_after: bool = True,\n    eval_conf: float = 0.25,\n    eval_iou: float = 0.70,\n) -> dict[str, Any]:\n    from ultralytics import YOLO\n\n    ensure_project_import_paths()\n    register_all_custom_attention_modules()\n    if boundary_loss_weight is not None:\n        enable_boundary_aware_mask_loss(boundary_loss_weight)\n\n    candidate = candidate_by_key(candidate_key)\n    exp_output = Path(output_root) / experiment_key\n    yaml_row = create_variant_yaml(\n        candidate_key=candidate_key,\n        variant_key=variant_key,\n        output_dir=exp_output,\n        nm=nm,\n        upsample_mode=upsample_mode,\n    )\n    model_spec = yaml_row["variant_yaml"]\n    variant_module_key = f"{candidate_key}_{variant_key}"\n\n    dataset_dir = exp_output / "datasets" / variant_module_key / f"seed_{seed}" / "dataset"\n    dataset_dir = copy_dataset_snapshot(Path(base_data_dir), dataset_dir, refresh=False)\n    data_yaml = write_data_yaml(Path(source_data_yaml), dataset_dir, dataset_dir / "data.yaml")\n\n    run_dir = run_dir_for(Path(runs_root), experiment_key, variant_module_key, seed)\n    best_path = run_dir / "weights" / "best.pt"\n    train_csv = exp_output / "reports" / f"{experiment_key}_train_runs.csv"\n    eval_csv = exp_output / "reports" / f"{experiment_key}_eval_results.csv"\n\n    train_row: dict[str, Any] = {\n        "experiment_key": experiment_key,\n        "variant_key": variant_key,\n        "module_key": candidate_key,\n        "variant_module_key": variant_module_key,\n        "display_name": candidate.get("display_name", candidate_key),\n        "seed": seed,\n        "model_spec": model_spec,\n        "source_yaml": yaml_row["source_yaml"],\n        "dataset_dir": str(dataset_dir),\n        "data_yaml": str(data_yaml),\n        "run_dir": str(run_dir),\n        "best_checkpoint": str(best_path),\n        "smoke": smoke,\n        "nm": nm,\n        "upsample_mode": upsample_mode,\n        "boundary_loss_weight": boundary_loss_weight,\n        **{f"yaml_{k}": v for k, v in yaml_row.items() if k not in {"candidate_key", "variant_key"}},\n    }\n\n    if best_path.exists():\n        train_row.update({"status": "skipped_existing", "reason": "best checkpoint already exists"})\n    else:\n        disable_ultralytics_albumentations()\n        yolo = YOLO(model_spec)\n        if load_pretrained_weights:\n            yolo = yolo.load(str(pretrained_weights))\n\n        actual_imgsz = 320 if smoke else int(imgsz)\n        actual_epochs = 1 if smoke else int(epochs)\n        actual_batch = 8 if smoke else int(batch)\n        actual_patience = 1 if smoke else int(patience)\n        clean_train_args = train_args_without_removed_keys(train_args)\n\n        start = time.time()\n        yolo.train(\n            data=str(data_yaml),\n            task="segment",\n            imgsz=actual_imgsz,\n            epochs=actual_epochs,\n            batch=actual_batch,\n            patience=actual_patience,\n            seed=seed,\n            project=str(run_dir.parent),\n            name=run_dir.name,\n            exist_ok=False,\n            pretrained=True,\n            plots=not smoke,\n            verbose=True,\n            save_period=save_period,\n            **clean_train_args,\n        )\n        train_row.update(\n            {\n                "status": "trained",\n                "train_time_min": round((time.time() - start) / 60.0, 4),\n                "imgsz": actual_imgsz,\n                "epochs": actual_epochs,\n                "batch": actual_batch,\n                "patience": actual_patience,\n                "save_period": save_period,\n                "train_args_json": json.dumps(clean_train_args, sort_keys=True),\n            }\n        )\n\n    train_row.update(read_best_epoch_from_results(run_dir))\n    append_csv_row(train_csv, train_row)\n\n    if evaluate_after and best_path.exists():\n        eval_row = evaluate_checkpoint(\n            candidate={**candidate, "key": variant_module_key, "augmentation": experiment_key},\n            checkpoint=best_path,\n            dataset_dir=dataset_dir,\n            data_yaml=data_yaml,\n            output_root=exp_output,\n            imgsz=320 if smoke else int(imgsz),\n            conf=eval_conf,\n            iou=eval_iou,\n            augment=False,\n            run_val=True,\n            plots=False,\n        )\n        eval_row.update(\n            {\n                "experiment_key": experiment_key,\n                "variant_key": variant_key,\n                "module_key": candidate_key,\n                "variant_module_key": variant_module_key,\n                "seed": seed,\n                "protocol_train_status": train_row.get("status"),\n            }\n        )\n        append_csv_row(eval_csv, eval_row)\n        train_row["eval_csv"] = str(eval_csv)\n\n    train_row["train_csv"] = str(train_csv)\n    return train_row\n', 'D': '"""Controlled architecture-attention helpers for custom-attention Group D.\n\nGroup D runs architecture ablations after A/B/C:\n- D1: SimAM at backbone P3/P4.\n- D2: CoordAtt at backbone P3/P4.\n- D3: lightweight LKA head refinement before Segment.\n\nThe helper reuses Group A evaluation and Group B training defaults, but keeps\nall new architecture registration process-local. It does not edit vendored\nUltralytics files or existing notebooks.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport importlib.util\nimport json\nimport math\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nimport torch.nn as nn\nimport yaml\n\n\nDEFAULT_IMGSZ = 640\nDEFAULT_BATCH = 16\nDEFAULT_SEED = 42\nDEFAULT_EPOCHS = 100\nDEFAULT_PATIENCE = 30\n\nBACKBONE_ATTENTION_TARGETS = [4, 6]\n\nSIMAM_BACKBONE_MODULE_KEYS = [\n    "triplet_attention_segment_head",\n    "cote_gate",\n    "sge_eca_head_gate",\n    "lpsc_soft_gate",\n]\n\nCA_BACKBONE_MODULE_KEYS = [\n    "triplet_attention_segment_head",\n    "cote_gate",\n]\n\nLKA_HEAD_MODULE_KEYS = [\n    "triplet_attention_segment_head",\n    "cote_gate",\n]\n\nEXTRA_CANDIDATES: dict[str, dict[str, Any]] = {\n    "lpsc_soft_gate": {\n        "key": "lpsc_soft_gate",\n        "display_name": "LPSC Soft Gate",\n        "family": "custom_attention_research_modules",\n        "augmentation": "clean_light",\n        "model_yaml": "custom_attention_research_modules/lpsc_soft_gate/generated_yamls/lpsc_soft_strong_augmentation.yaml",\n        "checkpoint_patterns": [\n            "/kaggle/working/runs/custom_attention_research_modules/lpsc_soft_strong_augmentation/weights/best.pt",\n        ],\n    },\n}\n\nSHORT_KEY_NAMES = {\n    "triplet_attention_segment_head": "triplet",\n    "cote_gate": "cote",\n    "sge_eca_head_gate": "sge_eca",\n    "lpsc_soft_gate": "lpsc_soft",\n    "simam_backbone_p3p4": "simam_p3p4",\n    "ca_backbone_dual": "ca_p3p4",\n    "lka_head_refine": "lka_head",\n}\n\n\ndef attention_root() -> Path:\n    here = Path(__file__).resolve()\n    for parent in [*here.parents, Path.cwd(), Path.cwd() / "yolov11n_attention"]:\n        if (parent / "custom_attention").exists() and (parent / "yolov11n_custom_attention_NhomA").exists():\n            return parent\n    return here.parents[2]\n\n\ndef _load_module_from_file(module_name: str, path: Path):\n    if module_name in sys.modules:\n        return sys.modules[module_name]\n    if not path.exists():\n        raise FileNotFoundError(f"Required helper not found: {path}")\n    spec = importlib.util.spec_from_file_location(module_name, path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Could not load helper from {path}")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[module_name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_group_a_utils():\n    root = attention_root()\n    path = root / "yolov11n_custom_attention_NhomA" / "_shared" / "nhomA_eval_utils.py"\n    return _load_module_from_file("nhomA_eval_utils_for_nhomD", path)\n\n\ndef load_group_b_utils():\n    root = attention_root()\n    path = root / "yolov11n_custom_attention_NhomB" / "_shared" / "nhomB_train_utils.py"\n    return _load_module_from_file("nhomB_train_utils_for_nhomD", path)\n\n\ndef ensure_project_import_paths() -> Path:\n    return load_group_a_utils().ensure_project_import_paths()\n\n\ndef resolve_existing_path(path_value: str | Path | None) -> Path | None:\n    return load_group_a_utils().resolve_existing_path(path_value)\n\n\ndef copy_dataset_snapshot(source_dataset: Path, snapshot_dir: Path, refresh: bool = False) -> Path:\n    return load_group_a_utils().copy_dataset_snapshot(source_dataset, snapshot_dir, refresh=refresh)\n\n\ndef write_data_yaml(source_data_yaml: Path, dataset_dir: Path, yaml_path: Path) -> Path:\n    return load_group_a_utils().write_data_yaml(source_data_yaml, dataset_dir, yaml_path)\n\n\ndef append_csv_row(path: Path, row: dict[str, Any]) -> None:\n    load_group_a_utils().append_csv_row(path, row)\n\n\ndef disable_ultralytics_albumentations() -> None:\n    load_group_a_utils().disable_ultralytics_albumentations()\n\n\ndef evaluate_checkpoint(*args, **kwargs) -> dict[str, Any]:\n    register_group_d_modules()\n    return load_group_a_utils().evaluate_checkpoint(*args, **kwargs)\n\n\ndef read_best_epoch_from_results(run_path: Path) -> dict[str, Any]:\n    return load_group_b_utils().read_best_epoch_from_results(run_path)\n\n\ndef train_args_without_removed_keys(train_args: dict[str, Any]) -> dict[str, Any]:\n    return load_group_b_utils().train_args_without_removed_keys(train_args)\n\n\ndef clean_light_train_args() -> dict[str, Any]:\n    return copy.deepcopy(load_group_b_utils().CLEAN_LIGHT_AUG_TRAIN_ARGS)\n\n\ndef stable_train_control() -> dict[str, Any]:\n    return copy.deepcopy(load_group_b_utils().STABLE_TRAIN_CONTROL)\n\n\ndef run_dir_for(runs_root: Path, experiment_key: str, module_key: str, seed: int) -> Path:\n    return load_group_b_utils().run_dir_for(Path(runs_root), experiment_key, module_key, seed)\n\n\ndef candidate_by_key(key: str) -> dict[str, Any]:\n    if key in EXTRA_CANDIDATES:\n        return dict(EXTRA_CANDIDATES[key])\n    return load_group_a_utils().candidate_by_key(key)\n\n\ndef selected_candidates(module_keys: list[str]) -> list[dict[str, Any]]:\n    return [candidate_by_key(key) for key in module_keys]\n\n\ndef _resolve_channels(c1: int | None = None, channels: int | None = None) -> int:\n    value = channels if channels is not None else c1\n    if value is None:\n        raise ValueError("A channel count must be provided.")\n    value = int(value)\n    if value <= 0:\n        raise ValueError(f"channels must be positive, got {value}")\n    return value\n\n\ndef _safe_groups(channels: int, preferred_groups: int) -> int:\n    channels = int(channels)\n    preferred_groups = max(1, int(preferred_groups))\n    if channels % preferred_groups == 0:\n        return preferred_groups\n    return max(1, math.gcd(channels, preferred_groups))\n\n\ndef _odd_kernel(kernel_size: int) -> int:\n    kernel_size = int(kernel_size)\n    return kernel_size if kernel_size % 2 else kernel_size + 1\n\n\nclass NAMGate(nn.Module):\n    """Normalization-aware gate returning [B, C, H, W]."""\n\n    def __init__(self, c1: int | None = None, groups: int = 16, eps: float = 1e-6, channels: int | None = None):\n        super().__init__()\n        channels = _resolve_channels(c1, channels)\n        self.norm = nn.GroupNorm(_safe_groups(channels, groups), channels, affine=True)\n        self.eps = float(eps)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        y = self.norm(x)\n        weight = self.norm.weight.abs()\n        weight = weight / (weight.sum() + self.eps)\n        return torch.sigmoid(y * weight.view(1, -1, 1, 1))\n\n\nclass SimAMGate(nn.Module):\n    """Parameter-free SimAM gate returning [B, C, H, W]."""\n\n    def __init__(self, c1: int | None = None, eps: float = 1e-4, channels: int | None = None):\n        super().__init__()\n        _resolve_channels(c1, channels)\n        self.eps = float(eps)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        n = x.shape[2] * x.shape[3] - 1\n        if n <= 0:\n            return torch.ones_like(x)\n        centered = (x - x.mean(dim=(2, 3), keepdim=True)).pow(2)\n        denom = 4.0 * (centered.sum(dim=(2, 3), keepdim=True) / n + self.eps)\n        return torch.sigmoid(centered / denom + 0.5)\n\n\nclass LPSCSoftGate(nn.Module):\n    """Soft LPSC variant used by the existing lpsc_soft YAML."""\n\n    def __init__(\n        self,\n        c1: int,\n        simam_eps: float = 1e-4,\n        nam_weight_init: float = 0.25,\n        eta_init: float = 1.25,\n        delta_init: float = 0.25,\n        gamma_init: float = 0.0,\n    ):\n        super().__init__()\n        channels = _resolve_channels(c1)\n        self.nam = NAMGate(channels)\n        self.simam = SimAMGate(channels, eps=simam_eps)\n        self.dw = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)\n        self.smooth = nn.AvgPool2d(kernel_size=3, stride=1, padding=1)\n        self.nam_weight = nn.Parameter(torch.tensor(float(nam_weight_init)))\n        self.eta = nn.Parameter(torch.tensor(float(eta_init)))\n        self.delta = nn.Parameter(torch.tensor(float(delta_init)))\n        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        identity = x\n        nam = self.nam(x)\n        simam = self.simam(x)\n        prior = self.dw(x)\n        prior = torch.sigmoid(prior - self.smooth(prior))\n        gate = torch.sigmoid(self.nam_weight * nam + self.eta * simam + self.delta * prior)\n        return identity + self.gamma * identity * gate\n\n\nclass LKAResidualAttention(nn.Module):\n    """Large-kernel residual attention that preserves [B, C, H, W]."""\n\n    def __init__(\n        self,\n        c1: int,\n        kernel_size: int = 5,\n        dilated_kernel_size: int = 7,\n        dilation: int = 3,\n        gamma_init: float = 0.0,\n    ):\n        super().__init__()\n        channels = _resolve_channels(c1)\n        kernel_size = _odd_kernel(kernel_size)\n        dilated_kernel_size = _odd_kernel(dilated_kernel_size)\n        dilation = max(1, int(dilation))\n        self.dw = nn.Conv2d(\n            channels,\n            channels,\n            kernel_size=kernel_size,\n            padding=(kernel_size - 1) // 2,\n            groups=channels,\n            bias=False,\n        )\n        self.dw_dilated = nn.Conv2d(\n            channels,\n            channels,\n            kernel_size=dilated_kernel_size,\n            padding=dilation * ((dilated_kernel_size - 1) // 2),\n            dilation=dilation,\n            groups=channels,\n            bias=False,\n        )\n        self.pw = nn.Conv2d(channels, channels, kernel_size=1, bias=False)\n        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        gate = torch.sigmoid(self.pw(self.dw_dilated(self.dw(x))))\n        return x + self.gamma * x * (gate - 0.5)\n\n\ndef register_group_d_modules() -> None:\n    """Register D-only modules after Group A registration has patched parse_model."""\n\n    ensure_project_import_paths()\n    load_group_a_utils().register_all_custom_attention_modules()\n    import ultralytics.nn.tasks as tasks\n\n    extra_modules = (LPSCSoftGate, LKAResidualAttention)\n    for module_cls in extra_modules:\n        setattr(tasks, module_cls.__name__, module_cls)\n        setattr(sys.modules["__main__"], module_cls.__name__, module_cls)\n\n    existing = tuple(getattr(tasks, "NHOMA_SINGLE_INPUT_ATTENTION_MODULES", ()))\n    merged = list(existing)\n    for module_cls in extra_modules:\n        if module_cls not in merged:\n            merged.append(module_cls)\n    tasks.NHOMA_SINGLE_INPUT_ATTENTION_MODULES = tuple(merged)\n\n    try:\n        torch.serialization.add_safe_globals(list(extra_modules))\n    except Exception:\n        pass\n\n\ndef _module_name(module_value: Any) -> str:\n    if isinstance(module_value, str):\n        return module_value\n    return getattr(module_value, "__name__", str(module_value))\n\n\ndef short_key(value: str) -> str:\n    text = str(value)\n    return SHORT_KEY_NAMES.get(text, text.replace("attention_segment_head", "att").replace("_backbone", "_bb"))\n\n\ndef source_yaml_for_candidate(candidate_key: str) -> Path:\n    candidate = candidate_by_key(candidate_key)\n    model_yaml = candidate.get("model_yaml")\n    if not model_yaml or str(model_yaml).endswith(".pt"):\n        raise ValueError(f"Candidate {candidate_key!r} does not expose a source YAML.")\n    resolved = resolve_existing_path(model_yaml)\n    if resolved is None:\n        raise FileNotFoundError(f"Model YAML not found for {candidate_key}: {model_yaml}")\n    return resolved\n\n\ndef _remap_from_value(value: Any, original_to_new: dict[int, int], target_to_inserted: dict[int, int]) -> Any:\n    if isinstance(value, int):\n        if value < 0:\n            return value\n        if value in target_to_inserted:\n            return target_to_inserted[value]\n        return original_to_new[value]\n    if isinstance(value, list):\n        return [_remap_from_value(item, original_to_new, target_to_inserted) for item in value]\n    return value\n\n\ndef insert_attention_after_backbone_indices(\n    cfg: dict[str, Any],\n    target_indices: list[int],\n    module_name: str,\n    module_args: list[Any] | None = None,\n) -> list[dict[str, Any]]:\n    """Insert a single-input module after original backbone layers and remap positive refs."""\n\n    module_args = list(module_args or [])\n    backbone = copy.deepcopy(cfg.get("backbone", []))\n    head = copy.deepcopy(cfg.get("head", []))\n    backbone_count = len(backbone)\n    layers = backbone + head\n    targets = set(int(i) for i in target_indices)\n    invalid = [idx for idx in targets if idx < 0 or idx >= backbone_count]\n    if invalid:\n        raise ValueError(f"Backbone insertion targets out of range: {invalid}; backbone_count={backbone_count}")\n\n    new_layers: list[list[Any]] = []\n    inserted_indices: set[int] = set()\n    original_to_new: dict[int, int] = {}\n    target_to_inserted: dict[int, int] = {}\n    changes: list[dict[str, Any]] = []\n\n    for original_index, layer in enumerate(layers):\n        original_to_new[original_index] = len(new_layers)\n        new_layers.append(copy.deepcopy(layer))\n        if original_index in targets:\n            inserted_index = len(new_layers)\n            new_layers.append([-1, 1, module_name, list(module_args)])\n            inserted_indices.add(inserted_index)\n            target_to_inserted[original_index] = inserted_index\n            changes.append(\n                {\n                    "target_original_index": original_index,\n                    "target_new_index": original_to_new[original_index],\n                    "inserted_index": inserted_index,\n                    "module": module_name,\n                    "args": list(module_args),\n                }\n            )\n\n    for new_index, layer in enumerate(new_layers):\n        if new_index in inserted_indices:\n            continue\n        layer[0] = _remap_from_value(layer[0], original_to_new, target_to_inserted)\n\n    new_backbone_count = backbone_count + len(targets)\n    cfg["backbone"] = new_layers[:new_backbone_count]\n    cfg["head"] = new_layers[new_backbone_count:]\n    return changes\n\n\ndef insert_lka_on_segment_inputs(\n    cfg: dict[str, Any],\n    module_args: list[Any] | None = None,\n) -> list[dict[str, Any]]:\n    """Insert LKAResidualAttention on each Segment input and rewire Segment."""\n\n    module_args = list(module_args or [5, 7, 3, 0.0])\n    backbone = copy.deepcopy(cfg.get("backbone", []))\n    head = copy.deepcopy(cfg.get("head", []))\n    layers = backbone + head\n    backbone_count = len(backbone)\n    segment_index = None\n    for index, layer in enumerate(layers):\n        if isinstance(layer, list) and len(layer) >= 4 and _module_name(layer[2]) == "Segment":\n            segment_index = index\n    if segment_index is None:\n        raise ValueError("No Segment layer found for LKA head insertion.")\n    if segment_index != len(layers) - 1:\n        raise ValueError("LKA insertion currently expects Segment to be the final model layer.")\n\n    prefix = copy.deepcopy(layers[:segment_index])\n    segment = copy.deepcopy(layers[segment_index])\n    original_from = segment[0]\n    from_list = original_from if isinstance(original_from, list) else [original_from]\n    new_from: list[int] = []\n    changes: list[dict[str, Any]] = []\n\n    for source_index in from_list:\n        if not isinstance(source_index, int) or source_index < 0:\n            raise ValueError(f"Segment input must be a non-negative layer index, got {source_index!r}")\n        inserted_index = len(prefix)\n        prefix.append([source_index, 1, "LKAResidualAttention", list(module_args)])\n        new_from.append(inserted_index)\n        changes.append(\n            {\n                "segment_original_input": source_index,\n                "inserted_index": inserted_index,\n                "module": "LKAResidualAttention",\n                "args": list(module_args),\n            }\n        )\n\n    segment[0] = new_from if isinstance(original_from, list) else new_from[0]\n    new_layers = prefix + [segment]\n    cfg["backbone"] = new_layers[:backbone_count]\n    cfg["head"] = new_layers[backbone_count:]\n    return changes\n\n\ndef create_variant_yaml(\n    candidate_key: str,\n    variant_key: str,\n    output_dir: Path,\n    variant_kind: str,\n    backbone_targets: list[int] | None = None,\n    lka_args: list[Any] | None = None,\n) -> dict[str, Any]:\n    source_yaml = source_yaml_for_candidate(candidate_key)\n    with source_yaml.open("r", encoding="utf-8") as f:\n        cfg = yaml.safe_load(f)\n    if not isinstance(cfg, dict):\n        raise ValueError(f"YAML did not parse to a mapping: {source_yaml}")\n\n    if variant_kind == "simam_backbone_p3p4":\n        changes = insert_attention_after_backbone_indices(\n            cfg,\n            backbone_targets or BACKBONE_ATTENTION_TARGETS,\n            "SimAM",\n            [],\n        )\n    elif variant_kind == "ca_backbone_dual":\n        changes = insert_attention_after_backbone_indices(\n            cfg,\n            backbone_targets or BACKBONE_ATTENTION_TARGETS,\n            "CoordAtt",\n            [],\n        )\n    elif variant_kind == "lka_head_refine":\n        changes = insert_lka_on_segment_inputs(cfg, module_args=lka_args or [5, 7, 3, 0.0])\n    else:\n        raise ValueError(f"Unknown Group D variant_kind: {variant_kind}")\n\n    out_dir = Path(output_dir) / "generated_yamls" / short_key(candidate_key)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    out_yaml = out_dir / f"{short_key(candidate_key)}_{short_key(variant_key)}.yaml"\n    out_yaml.parent.mkdir(parents=True, exist_ok=True)\n    with out_yaml.open("w", encoding="utf-8") as f:\n        yaml.safe_dump(cfg, f, sort_keys=False)\n\n    return {\n        "candidate_key": candidate_key,\n        "variant_key": variant_key,\n        "variant_kind": variant_kind,\n        "source_yaml": str(source_yaml),\n        "variant_yaml": str(out_yaml),\n        "change_count": len(changes),\n        "change_details": json.dumps(changes, sort_keys=True),\n    }\n\n\ndef create_variant_yamls(\n    module_keys: list[str],\n    variant_key: str,\n    output_dir: Path,\n    variant_kind: str,\n    backbone_targets: list[int] | None = None,\n    lka_args: list[Any] | None = None,\n) -> list[dict[str, Any]]:\n    return [\n        create_variant_yaml(\n            candidate_key=module_key,\n            variant_key=variant_key,\n            output_dir=output_dir,\n            variant_kind=variant_kind,\n            backbone_targets=backbone_targets,\n            lka_args=lka_args,\n        )\n        for module_key in module_keys\n    ]\n\n\ndef _shape_summary(value: Any) -> Any:\n    if hasattr(value, "shape"):\n        return tuple(int(v) for v in value.shape)\n    if isinstance(value, (list, tuple)):\n        return [_shape_summary(item) for item in value]\n    if isinstance(value, dict):\n        return {key: _shape_summary(item) for key, item in value.items()}\n    return type(value).__name__\n\n\ndef sanity_check_model_yaml(model_yaml: str | Path, imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> dict[str, Any]:\n    from ultralytics import YOLO\n\n    register_group_d_modules()\n    model_path = resolve_existing_path(model_yaml) or Path(model_yaml)\n    if not model_path.exists():\n        raise FileNotFoundError(f"Model YAML not found: {model_yaml}")\n    yolo = YOLO(str(model_path))\n    try:\n        net = yolo.model.to(device).eval()\n        x = torch.zeros(1, 3, imgsz, imgsz, device=device)\n        with torch.no_grad():\n            output = net(x)\n        shapes = _shape_summary(output)\n    except Exception as exc:  # noqa: BLE001\n        raise RuntimeError(f"Dummy forward failed for {model_path}: {exc}") from exc\n\n    return {\n        "model_yaml": str(model_path),\n        "imgsz": imgsz,\n        "device": device,\n        "status": "ok",\n        "output_shapes": shapes,\n    }\n\n\ndef sanity_check_variant_yamls(rows: list[dict[str, Any]], imgsz: int = DEFAULT_IMGSZ, device: str = "cpu") -> list[dict[str, Any]]:\n    checked = []\n    for row in rows:\n        item = dict(row)\n        try:\n            item.update(sanity_check_model_yaml(item["variant_yaml"], imgsz=imgsz, device=device))\n        except Exception as exc:  # noqa: BLE001\n            item.update({"status": "error", "error_type": exc.__class__.__name__, "error": str(exc)})\n        checked.append(item)\n    return checked\n\n\ndef run_arch_variant_experiment(\n    candidate_key: str,\n    experiment_key: str,\n    variant_key: str,\n    variant_kind: str,\n    base_data_dir: Path,\n    source_data_yaml: Path,\n    output_root: Path,\n    runs_root: Path,\n    train_args: dict[str, Any],\n    seed: int = DEFAULT_SEED,\n    imgsz: int = DEFAULT_IMGSZ,\n    epochs: int = DEFAULT_EPOCHS,\n    batch: int = DEFAULT_BATCH,\n    patience: int = DEFAULT_PATIENCE,\n    smoke: bool = False,\n    save_period: int = -1,\n    pretrained_weights: str = "yolo11n-seg.pt",\n    load_pretrained_weights: bool = True,\n    evaluate_after: bool = True,\n    eval_conf: float = 0.25,\n    eval_iou: float = 0.70,\n) -> dict[str, Any]:\n    from ultralytics import YOLO\n\n    register_group_d_modules()\n    candidate = candidate_by_key(candidate_key)\n    exp_output = Path(output_root) / experiment_key\n    yaml_row = create_variant_yaml(\n        candidate_key=candidate_key,\n        variant_key=variant_key,\n        output_dir=exp_output,\n        variant_kind=variant_kind,\n    )\n    model_spec = yaml_row["variant_yaml"]\n    variant_module_key = f"{candidate_key}_{variant_key}"\n\n    dataset_dir = exp_output / "datasets" / variant_module_key / f"seed_{seed}" / "dataset"\n    dataset_dir = copy_dataset_snapshot(Path(base_data_dir), dataset_dir, refresh=False)\n    data_yaml = write_data_yaml(Path(source_data_yaml), dataset_dir, dataset_dir / "data.yaml")\n\n    run_dir = run_dir_for(Path(runs_root), experiment_key, variant_module_key, seed)\n    best_path = run_dir / "weights" / "best.pt"\n    train_csv = exp_output / "reports" / f"{experiment_key}_train_runs.csv"\n    eval_csv = exp_output / "reports" / f"{experiment_key}_eval_results.csv"\n\n    train_row: dict[str, Any] = {\n        "experiment_key": experiment_key,\n        "variant_key": variant_key,\n        "variant_kind": variant_kind,\n        "module_key": candidate_key,\n        "variant_module_key": variant_module_key,\n        "display_name": candidate.get("display_name", candidate_key),\n        "seed": seed,\n        "model_spec": model_spec,\n        "source_yaml": yaml_row["source_yaml"],\n        "change_count": yaml_row["change_count"],\n        "change_details": yaml_row["change_details"],\n        "dataset_dir": str(dataset_dir),\n        "data_yaml": str(data_yaml),\n        "run_dir": str(run_dir),\n        "best_checkpoint": str(best_path),\n        "smoke": smoke,\n    }\n\n    if best_path.exists():\n        train_row.update({"status": "skipped_existing", "reason": "best checkpoint already exists"})\n    else:\n        disable_ultralytics_albumentations()\n        yolo = YOLO(model_spec)\n        if load_pretrained_weights:\n            yolo = yolo.load(str(pretrained_weights))\n\n        actual_imgsz = 320 if smoke else int(imgsz)\n        actual_epochs = 1 if smoke else int(epochs)\n        actual_batch = 8 if smoke else int(batch)\n        actual_patience = 1 if smoke else int(patience)\n        clean_train_args = train_args_without_removed_keys(train_args)\n\n        start = time.time()\n        yolo.train(\n            data=str(data_yaml),\n            task="segment",\n            imgsz=actual_imgsz,\n            epochs=actual_epochs,\n            batch=actual_batch,\n            patience=actual_patience,\n            seed=seed,\n            project=str(run_dir.parent),\n            name=run_dir.name,\n            exist_ok=False,\n            pretrained=True,\n            plots=not smoke,\n            verbose=True,\n            save_period=save_period,\n            **clean_train_args,\n        )\n        train_row.update(\n            {\n                "status": "trained",\n                "train_time_min": round((time.time() - start) / 60.0, 4),\n                "imgsz": actual_imgsz,\n                "epochs": actual_epochs,\n                "batch": actual_batch,\n                "patience": actual_patience,\n                "save_period": save_period,\n                "train_args_json": json.dumps(clean_train_args, sort_keys=True),\n            }\n        )\n\n    train_row.update(read_best_epoch_from_results(run_dir))\n    append_csv_row(train_csv, train_row)\n\n    if evaluate_after and best_path.exists():\n        eval_row = evaluate_checkpoint(\n            candidate={**candidate, "key": variant_module_key, "augmentation": experiment_key},\n            checkpoint=best_path,\n            dataset_dir=dataset_dir,\n            data_yaml=data_yaml,\n            output_root=exp_output,\n            imgsz=320 if smoke else int(imgsz),\n            conf=eval_conf,\n            iou=eval_iou,\n            augment=False,\n            run_val=True,\n            plots=False,\n        )\n        eval_row.update(\n            {\n                "experiment_key": experiment_key,\n                "variant_key": variant_key,\n                "variant_kind": variant_kind,\n                "module_key": candidate_key,\n                "variant_module_key": variant_module_key,\n                "seed": seed,\n                "protocol_train_status": train_row.get("status"),\n            }\n        )\n        append_csv_row(eval_csv, eval_row)\n        train_row["eval_csv"] = str(eval_csv)\n\n    train_row["train_csv"] = str(train_csv)\n    return train_row\n'}
_EMBEDDED_MODEL_YAMLS = {'custom_attention/triplet_attention_segment_head/yolov11n_triplet_segment.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- - -1\n  - 1\n  - Conv\n  - - 16\n    - 3\n    - 2\n- - -1\n  - 1\n  - Conv\n  - - 32\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 256\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - -1\n  - 1\n  - SPPF\n  - - 256\n    - 5\n- - -1\n  - 1\n  - C2PSA\n  - - 256\nhead:\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 6\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 4\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - - -1\n    - 13\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - - -1\n    - 10\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - 16\n  - 1\n  - TripletAttention\n  - - 7\n    - 0.1\n- - 19\n  - 1\n  - TripletAttention\n  - - 7\n    - 0.1\n- - 22\n  - 1\n  - TripletAttention\n  - - 7\n    - 0.1\n- - - 23\n    - 24\n    - 25\n  - 1\n  - Segment\n  - - nc\n    - 32\n    - 64\n', 'custom_attention/cesa_lite_segment_head/yolov11n_cesa_lite.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- - -1\n  - 1\n  - Conv\n  - - 16\n    - 3\n    - 2\n- - -1\n  - 1\n  - Conv\n  - - 32\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 256\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - -1\n  - 1\n  - SPPF\n  - - 256\n    - 5\n- - -1\n  - 1\n  - C2PSA\n  - - 256\nhead:\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 6\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 4\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - - -1\n    - 13\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - - -1\n    - 10\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - 16\n  - 1\n  - CESALite\n  - - 32\n    - 3\n    - 0.1\n- - 19\n  - 1\n  - CESALite\n  - - 32\n    - 3\n    - 0.1\n- - 22\n  - 1\n  - CESALite\n  - - 32\n    - 3\n    - 0.1\n- - - 23\n    - 24\n    - 25\n  - 1\n  - Segment\n  - - nc\n    - 32\n    - 64\n', 'custom_attention/sge_eca_head_gate/model.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- - -1\n  - 1\n  - Conv\n  - - 16\n    - 3\n    - 2\n- - -1\n  - 1\n  - Conv\n  - - 32\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 256\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - -1\n  - 1\n  - SPPF\n  - - 256\n    - 5\n- - -1\n  - 1\n  - C2PSA\n  - - 256\nhead:\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 6\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 4\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - - -1\n    - 13\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - - -1\n    - 10\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - 16\n  - 1\n  - SGEECAHeadGate\n  - - 8\n    - 3\n    - 0.0\n- - 19\n  - 1\n  - SGEECAHeadGate\n  - - 8\n    - 3\n    - 0.0\n- - - 23\n    - 24\n    - 22\n  - 1\n  - Segment\n  - - nc\n    - 32\n    - 64\n', 'custom_attention_research_modules/cote_gate/cote_gate.yaml': '# YOLO11n-seg with CoTEGate inserted P4-only before Segment.\nnc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n  - [-1, 1, Conv, [16, 3, 2]]\n  - [-1, 1, Conv, [32, 3, 2]]\n  - [-1, 1, C3k2, [64, false, 0.25]]\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [-1, 1, C3k2, [64, false, 0.25]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [-1, 1, C3k2, [128, false, 0.25]]\n  - [-1, 1, Conv, [256, 3, 2]]\n  - [-1, 1, C3k2, [256, true]]\n  - [-1, 1, SPPF, [256, 5]]\n  - [-1, 1, C2PSA, [256]]\nhead:\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 6], 1, Concat, [1]]\n  - [-1, 1, C3k2, [128, false]]\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 4], 1, Concat, [1]]\n  - [-1, 1, C3k2, [64, false]]\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [[-1, 13], 1, Concat, [1]]\n  - [-1, 1, C3k2, [128, false]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [[-1, 10], 1, Concat, [1]]\n  - [-1, 1, C3k2, [256, true]]\n  - [19, 1, CoTEGate, [3, 5, 0.20, 0.0]]\n  - [[16, 23, 22], 1, Segment, [nc, 32, 64]]\n', 'custom_attention_research_modules/cote_gate/generated_yamls/cote_gate_strong_augmentation.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- - -1\n  - 1\n  - Conv\n  - - 16\n    - 3\n    - 2\n- - -1\n  - 1\n  - Conv\n  - - 32\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 256\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - -1\n  - 1\n  - SPPF\n  - - 256\n    - 5\n- - -1\n  - 1\n  - C2PSA\n  - - 256\nhead:\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 6\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 4\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - - -1\n    - 13\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - - -1\n    - 10\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - 19\n  - 1\n  - CoTEGate\n  - - 3\n    - 5\n    - 0.2\n    - 0.0\n- - - 16\n    - 23\n    - 22\n  - 1\n  - Segment\n  - - nc\n    - 32\n    - 64\n', 'custom_attention_research_modules/lpsc_gate/lpsc_gate.yaml': '# YOLO11n-seg with LPSCGate inserted P4-only before Segment.\nnc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n  - [-1, 1, Conv, [16, 3, 2]]\n  - [-1, 1, Conv, [32, 3, 2]]\n  - [-1, 1, C3k2, [64, false, 0.25]]\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [-1, 1, C3k2, [64, false, 0.25]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [-1, 1, C3k2, [128, false, 0.25]]\n  - [-1, 1, Conv, [256, 3, 2]]\n  - [-1, 1, C3k2, [256, true]]\n  - [-1, 1, SPPF, [256, 5]]\n  - [-1, 1, C2PSA, [256]]\nhead:\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 6], 1, Concat, [1]]\n  - [-1, 1, C3k2, [128, false]]\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 4], 1, Concat, [1]]\n  - [-1, 1, C3k2, [64, false]]\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [[-1, 13], 1, Concat, [1]]\n  - [-1, 1, C3k2, [128, false]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [[-1, 10], 1, Concat, [1]]\n  - [-1, 1, C3k2, [256, true]]\n  - [19, 1, LPSCGate, [0.0001, 1.0, 0.5, 0.0]]\n  - [[16, 23, 22], 1, Segment, [nc, 32, 64]]\n', 'custom_attention_research_modules/lpsc_gate/generated_yamls/lpsc_gate_strong_augmentation.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- - -1\n  - 1\n  - Conv\n  - - 16\n    - 3\n    - 2\n- - -1\n  - 1\n  - Conv\n  - - 32\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 256\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - -1\n  - 1\n  - SPPF\n  - - 256\n    - 5\n- - -1\n  - 1\n  - C2PSA\n  - - 256\nhead:\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 6\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 4\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - - -1\n    - 13\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - - -1\n    - 10\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - 19\n  - 1\n  - LPSCGate\n  - - 0.0001\n    - 1.0\n    - 0.5\n    - 0.0\n- - - 16\n    - 23\n    - 22\n  - 1\n  - Segment\n  - - nc\n    - 32\n    - 64\n', 'custom_attention_research_modules/lpsc_soft_gate/generated_yamls/lpsc_soft_strong_augmentation.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- - -1\n  - 1\n  - Conv\n  - - 16\n    - 3\n    - 2\n- - -1\n  - 1\n  - Conv\n  - - 32\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n    - 0.25\n- - -1\n  - 1\n  - Conv\n  - - 256\n    - 3\n    - 2\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - -1\n  - 1\n  - SPPF\n  - - 256\n    - 5\n- - -1\n  - 1\n  - C2PSA\n  - - 256\nhead:\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 6\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - nn.Upsample\n  - - null\n    - 2\n    - nearest\n- - - -1\n    - 4\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 64\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 64\n    - 3\n    - 2\n- - - -1\n    - 13\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 128\n    - false\n- - -1\n  - 1\n  - Conv\n  - - 128\n    - 3\n    - 2\n- - - -1\n    - 10\n  - 1\n  - Concat\n  - - 1\n- - -1\n  - 1\n  - C3k2\n  - - 256\n    - true\n- - 19\n  - 1\n  - LPSCSoftGate\n  - - 0.0001\n    - 0.25\n    - 1.25\n    - 0.25\n    - 0.0\n- - - 16\n    - 23\n    - 22\n  - 1\n  - Segment\n  - - nc\n    - 32\n    - 64\n', '__baseline_yolo11n_seg_resolved.yaml': 'nc: 2\ndepth_multiple: 1.0\nwidth_multiple: 1.0\nbackbone:\n- [-1, 1, Conv, [16, 3, 2]]\n- [-1, 1, Conv, [32, 3, 2]]\n- [-1, 1, C3k2, [128, false, 0.25]]\n- [-1, 1, Conv, [64, 3, 2]]\n- [-1, 1, C3k2, [64, false, 0.25]]\n- [-1, 1, Conv, [128, 3, 2]]\n- [-1, 1, C3k2, [128, true]]\n- [-1, 1, Conv, [256, 3, 2]]\n- [-1, 1, C3k2, [256, true]]\n- [-1, 1, SPPF, [256, 5]]\n- [-1, 1, C2PSA, [256]]\nhead:\n- [-1, 1, nn.Upsample, [null, 2, nearest]]\n- [[-1, 6], 1, Concat, [1]]\n- [-1, 1, C3k2, [128, false]]\n- [-1, 1, nn.Upsample, [null, 2, nearest]]\n- [[-1, 4], 1, Concat, [1]]\n- [-1, 1, C3k2, [64, false]]\n- [-1, 1, Conv, [64, 3, 2]]\n- [[-1, 13], 1, Concat, [1]]\n- [-1, 1, C3k2, [128, false]]\n- [-1, 1, Conv, [128, 3, 2]]\n- [[-1, 10], 1, Concat, [1]]\n- [-1, 1, C3k2, [256, true]]\n- [[16, 19, 22], 1, Segment, [nc, 32, 64]]\n'}


def _exec_helper_module(module_name, source):
    module = types.ModuleType(module_name)
    module.__file__ = f"/kaggle/working/_standalone_runtime/{module_name}.py"
    sys.modules[module_name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    return module


_A = _exec_helper_module("standalone_nhomA_eval_utils", _HELPER_SOURCES["A"])
_B = _exec_helper_module("standalone_nhomB_train_utils", _HELPER_SOURCES["B"])
_C = _exec_helper_module("standalone_nhomC_mask_utils", _HELPER_SOURCES["C"])
_D = _exec_helper_module("standalone_nhomD_arch_utils", _HELPER_SOURCES["D"])


def _standalone_working_dir():
    kaggle_working = Path("/kaggle/working")
    return kaggle_working if kaggle_working.exists() else Path.cwd()


def _standalone_ensure_project_import_paths():
    roots = [Path.cwd(), _standalone_working_dir(), Path("/kaggle/input")]
    for root in roots:
        if root.exists():
            text = str(root)
            if text not in sys.path:
                sys.path.insert(0, text)
    if importlib.util.find_spec("ultralytics") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    return Path.cwd()


def _safe_yaml_name(key):
    text = str(key).replace("\\", "/").strip("/")
    text = text.replace("/", "__").replace(" ", "_")
    return text if text.endswith(".yaml") else f"{text}.yaml"


def _embedded_yaml_content(path_value):
    if path_value is None:
        return None, None
    text = str(path_value).replace("\\", "/")
    candidates = [text, text.lstrip("./")]
    candidates.append(Path(text).name)
    for key, value in _EMBEDDED_MODEL_YAMLS.items():
        norm_key = str(key).replace("\\", "/")
        if norm_key in candidates or text.endswith(norm_key) or Path(norm_key).name == Path(text).name:
            return norm_key, value
    return None, None


def _standalone_resolve_existing_path(path_value):
    if not path_value:
        return None
    text = str(path_value)
    path = Path(text)
    roots = [Path.cwd(), _standalone_working_dir(), Path("/kaggle/input"), Path.cwd() / "yolov11n_attention"]
    candidates = [path] if path.is_absolute() else [root / text for root in roots]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    key, content = _embedded_yaml_content(text)
    if content is not None:
        out_dir = _standalone_working_dir() / "standalone_model_yamls"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / _safe_yaml_name(key)
        if not out_path.exists() or out_path.read_text(encoding="utf-8") != content:
            out_path.write_text(content, encoding="utf-8")
        return out_path
    return None


# ----- Shape-preserving fallback attention modules -----
import torch
import torch.nn as nn
import torch.nn.functional as F


def _resolve_channels(c1=None, channels=None):
    value = channels if channels is not None else c1
    if value is None:
        raise ValueError("channels must be provided")
    value = int(value)
    if value <= 0:
        raise ValueError(f"channels must be positive, got {value}")
    return value


def _safe_groups(channels, preferred=8):
    channels = int(channels)
    preferred = max(1, int(preferred))
    if channels % preferred == 0:
        return preferred
    return max(1, math.gcd(channels, preferred))


def _odd(k):
    k = int(k)
    return k if k % 2 else k + 1


class _IdentityResidualAttention(nn.Module):
    def __init__(self, c1=None, *args, **kwargs):
        super().__init__()
        self.c1 = c1
    def forward(self, x):
        return x


class _SpatialGate(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        k = _odd(kernel_size)
        self.conv = nn.Conv2d(2, 1, k, padding=(k - 1) // 2, bias=False)
    def forward(self, x):
        return torch.sigmoid(self.conv(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], dim=1)))


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = float(e_lambda)
    def forward(self, x):
        n = x.shape[2] * x.shape[3] - 1
        if n <= 0:
            return x
        centered = (x - x.mean(dim=(2, 3), keepdim=True)).pow(2)
        y = centered / (4.0 * (centered.sum(dim=(2, 3), keepdim=True) / n + self.e_lambda)) + 0.5
        return x * torch.sigmoid(y)


class CoordAtt(nn.Module):
    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        c1 = _resolve_channels(c1)
        c2 = c1 if c2 is None else int(c2)
        mip = max(8, c1 // max(1, int(reduction)))
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(c1, mip, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, 1, bias=False)
        self.conv_w = nn.Conv2d(mip, c2, 1, bias=False)
    def forward(self, x):
        identity = x
        n, c, h, w = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return identity * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w))


class ECAGate(nn.Module):
    def __init__(self, c1=None, k_size=3, channels=None):
        super().__init__()
        _resolve_channels(c1, channels)
        k = _odd(k_size)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=(k - 1) // 2, bias=False)
    def forward(self, x):
        y = self.pool(x).squeeze(-1).transpose(1, 2).contiguous()
        y = self.conv(y).transpose(1, 2).unsqueeze(-1).contiguous()
        return torch.sigmoid(y)


class TripletAttention(nn.Module):
    def __init__(self, c1=None, kernel_size=7, gamma_init=0.1):
        super().__init__()
        self.cw_gate = _SpatialGate(kernel_size)
        self.hc_gate = _SpatialGate(kernel_size)
        self.hw_gate = _SpatialGate(kernel_size)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        x1 = (x.permute(0, 2, 1, 3).contiguous() * self.cw_gate(x.permute(0, 2, 1, 3).contiguous())).permute(0, 2, 1, 3).contiguous()
        x2p = x.permute(0, 3, 2, 1).contiguous()
        x2 = (x2p * self.hc_gate(x2p)).permute(0, 3, 2, 1).contiguous()
        x3 = x * self.hw_gate(x)
        return x + self.gamma * ((x1 + x2 + x3) / 3.0)


class CESALite(nn.Module):
    def __init__(self, c1, reduction=32, kernel_size=3, gamma_init=0.1):
        super().__init__()
        self.coord = CoordAtt(c1, c1, reduction)
        self.eca = ECAGate(c1, kernel_size)
        self.simam = SimAM(c1)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        y = self.coord(x) * self.eca(x).expand_as(x)
        y = self.simam(y)
        return x + self.gamma * (y - x)


class CALiteSpatialGate(nn.Module):
    def __init__(self, c1, reduction=32, kernel_size=7, gamma_init=0.1):
        super().__init__()
        self.coord = CoordAtt(c1, c1, reduction)
        self.spatial = _SpatialGate(kernel_size)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        return x + self.gamma * x * self.spatial(self.coord(x))


class NAMAttention(nn.Module):
    def __init__(self, c1, gamma_init=0.1, eps=1e-6):
        super().__init__()
        c1 = _resolve_channels(c1)
        self.norm = nn.GroupNorm(_safe_groups(c1, 16), c1, affine=True)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
        self.eps = float(eps)
    def forward(self, x):
        y = self.norm(x)
        weight = self.norm.weight.abs()
        weight = weight / (weight.sum() + self.eps)
        return x + self.gamma * x * torch.sigmoid(y * weight.view(1, -1, 1, 1))


class LowFPCBAMLite(nn.Module):
    def __init__(self, c1, kernel_size=7, gamma_init=0.1):
        super().__init__()
        self.spatial = _SpatialGate(kernel_size)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        return x + self.gamma * x * self.spatial(x)


class SGEECAHeadGate(nn.Module):
    def __init__(self, c1, groups=8, eca_kernel=3, gamma_init=0.0):
        super().__init__()
        c1 = _resolve_channels(c1)
        self.gn = nn.GroupNorm(_safe_groups(c1, groups), c1, affine=True)
        self.eca = ECAGate(c1, eca_kernel)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        gate = torch.sigmoid(self.gn(x)) * self.eca(x).expand_as(x)
        return x + self.gamma * x * (gate - 0.5)


class CoTEGate(nn.Module):
    def __init__(self, c1, eca_k=3, triplet_k=5, alpha_init=0.20, gamma_init=0.0):
        super().__init__()
        self.triplet = TripletAttention(c1, triplet_k, 1.0)
        self.eca = ECAGate(c1, eca_k)
        self.smooth = nn.AvgPool2d(3, stride=1, padding=1)
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        t = torch.sigmoid(self.triplet(x) - x)
        c = self.eca(x).expand_as(x)
        gate = torch.sigmoid(t + c - self.alpha * self.smooth(torch.abs(t - c)))
        return x + self.gamma * x * gate


class LPSCGate(nn.Module):
    def __init__(self, c1, simam_eps=1e-4, eta_init=1.0, delta_init=0.5, gamma_init=0.0):
        super().__init__()
        c1 = _resolve_channels(c1)
        self.nam = NAMAttention(c1, gamma_init=1.0)
        self.simam = SimAM(c1, simam_eps)
        self.dw = nn.Conv2d(c1, c1, 3, padding=1, groups=c1, bias=False)
        self.smooth = nn.AvgPool2d(3, stride=1, padding=1)
        self.eta = nn.Parameter(torch.tensor(float(eta_init)))
        self.delta = nn.Parameter(torch.tensor(float(delta_init)))
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        prior = torch.sigmoid(self.dw(x) - self.smooth(self.dw(x)))
        gate = torch.sigmoid((self.nam(x) - x) + self.eta * self.simam(x) + self.delta * prior)
        return x + self.gamma * x * gate


class LPSCSoftGate(LPSCGate):
    def __init__(self, c1, simam_eps=1e-4, nam_weight_init=0.25, eta_init=1.25, delta_init=0.25, gamma_init=0.0):
        super().__init__(c1, simam_eps, eta_init, delta_init, gamma_init)
        self.nam_weight = nn.Parameter(torch.tensor(float(nam_weight_init)))
    def forward(self, x):
        prior = torch.sigmoid(self.dw(x) - self.smooth(self.dw(x)))
        gate = torch.sigmoid(self.nam_weight * (self.nam(x) - x) + self.eta * self.simam(x) + self.delta * prior)
        return x + self.gamma * x * gate


class LKAResidualAttention(nn.Module):
    def __init__(self, c1, kernel_size=5, dilated_kernel_size=7, dilation=3, gamma_init=0.0):
        super().__init__()
        c1 = _resolve_channels(c1)
        k1 = _odd(kernel_size)
        k2 = _odd(dilated_kernel_size)
        dilation = max(1, int(dilation))
        self.dw = nn.Conv2d(c1, c1, k1, padding=(k1 - 1) // 2, groups=c1, bias=False)
        self.dw_dilated = nn.Conv2d(c1, c1, k2, padding=dilation * ((k2 - 1) // 2), dilation=dilation, groups=c1, bias=False)
        self.pw = nn.Conv2d(c1, c1, 1, bias=False)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        gate = torch.sigmoid(self.pw(self.dw_dilated(self.dw(x))))
        return x + self.gamma * x * (gate - 0.5)


class AttentionGate(nn.Module):
    def __init__(self, c1, gate_channels=None, inter_ratio=2, gamma_init=0.1):
        super().__init__()
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        if isinstance(x, (list, tuple)):
            x = x[0]
        return x


BoundaryAwareLiteAttention = _IdentityResidualAttention
CASpatialLowFPGate = _IdentityResidualAttention
CESLite = _IdentityResidualAttention
ContextSuppressionGateLite = _IdentityResidualAttention
LowFPResidualSpatialGate = _IdentityResidualAttention
P3P4SemanticAttentionGate = _IdentityResidualAttention
PrototypeAwareMaskGateLite = _IdentityResidualAttention
SCSGGate = _IdentityResidualAttention
CoLPSCGate = _IdentityResidualAttention
CoTESimAMGate = CoTEGate
CoTESimAMRescueGate = CoTEGate
CoTESoftDisagreementGate = CoTEGate


def _standalone_register_all_custom_attention_modules():
    _standalone_ensure_project_import_paths()
    import ultralytics.nn.tasks as tasks
    import __main__

    single_input_modules = (
        CESALite, TripletAttention, CALiteSpatialGate, NAMAttention, LowFPCBAMLite,
        SGEECAHeadGate, CESLite, LowFPResidualSpatialGate, P3P4SemanticAttentionGate,
        BoundaryAwareLiteAttention, ContextSuppressionGateLite, CASpatialLowFPGate,
        PrototypeAwareMaskGateLite, CoTEGate, LPSCGate, LPSCSoftGate, CoTESimAMGate,
        CoTESimAMRescueGate, CoTESoftDisagreementGate, SCSGGate, CoLPSCGate,
        CoordAtt, SimAM, LKAResidualAttention,
    )
    for cls in (*single_input_modules, AttentionGate, ECAGate):
        setattr(tasks, cls.__name__, cls)
        setattr(__main__, cls.__name__, cls)

    tasks.NHOMA_SINGLE_INPUT_ATTENTION_MODULES = single_input_modules
    tasks.AttentionGate = AttentionGate

    if not getattr(tasks, "_shrimp_standalone_attention_parse_patched", False):
        source = inspect.getsource(tasks.parse_model)
        if "NHOMA_SINGLE_INPUT_ATTENTION_MODULES" not in source:
            marker = "        elif m in frozenset(\n            {\n                Detect,"
            insert = """        elif m in NHOMA_SINGLE_INPUT_ATTENTION_MODULES:\n            c1 = ch[f]\n            c2 = c1\n            args = [c1, *args]\n        elif m is AttentionGate:\n            c1 = ch[f[0]]\n            cg = ch[f[1]]\n            c2 = c1\n            args = [c1, cg, *args]\n"""
            if marker in source:
                patched = source.replace(marker, insert + marker, 1)
                exec(compile(patched, "<shrimp_standalone_parse_model>", "exec"), tasks.__dict__)
            else:
                print("Warning: parse_model marker not found; relying on existing Ultralytics module branches.")
        tasks._shrimp_standalone_attention_parse_patched = True

    try:
        torch.serialization.add_safe_globals(list(single_input_modules) + [AttentionGate, ECAGate])
    except Exception:
        pass


_A.ensure_project_import_paths = _standalone_ensure_project_import_paths
_A.register_all_custom_attention_modules = _standalone_register_all_custom_attention_modules
_A.resolve_existing_path = _standalone_resolve_existing_path

_B.load_group_a_utils = lambda: _A
_C.load_group_a_utils = lambda: _A
_C.load_group_b_utils = lambda: _B
_D.load_group_a_utils = lambda: _A
_D.load_group_b_utils = lambda: _B


def _standalone_candidate_by_key(key):
    if hasattr(_D, "EXTRA_CANDIDATES") and key in _D.EXTRA_CANDIDATES:
        return dict(_D.EXTRA_CANDIDATES[key])
    return _A.candidate_by_key(key)


def _standalone_source_yaml_for_candidate(candidate_key):
    if candidate_key == "baseline_clean":
        path = _standalone_resolve_existing_path("__baseline_yolo11n_seg_resolved.yaml")
        if path is None:
            raise FileNotFoundError("Embedded baseline YAML is missing.")
        return path
    candidate = _standalone_candidate_by_key(candidate_key)
    model_yaml = candidate.get("model_yaml")
    if not model_yaml or str(model_yaml).endswith(".pt"):
        raise ValueError(f"Candidate {candidate_key!r} does not expose a source YAML.")
    path = _standalone_resolve_existing_path(model_yaml)
    if path is None:
        raise FileNotFoundError(f"Model YAML not found for {candidate_key}: {model_yaml}")
    return path


_C.source_yaml_for_candidate = _standalone_source_yaml_for_candidate
_D.source_yaml_for_candidate = _standalone_source_yaml_for_candidate

_MODULE_FOR_GROUP = {"A": _A, "B": _B, "C": _C, "D": _D}[_STANDALONE_GROUP]
for _name, _value in _MODULE_FOR_GROUP.__dict__.items():
    if not _name.startswith("__"):
        globals()[_name] = _value

# Shared names useful in existing notebook cells.
globals().update({
    "pd": pd,
    "display": display,
    "Path": Path,
    "register_all_custom_attention_modules": _standalone_register_all_custom_attention_modules
        if _STANDALONE_GROUP in {"A", "B"} else globals().get("register_all_custom_attention_modules", _standalone_register_all_custom_attention_modules),
})

print(f"Standalone runtime loaded for Group {_STANDALONE_GROUP}. No external _shared helper files are required.")


### Experiment Configuration


In [ ]:
BASE_DATA_DIR = Path('/kaggle/working/shrimpDisHandSegV2-1')
DATA_YAML = BASE_DATA_DIR / 'data.yaml'
OUTPUT_ROOT = Path('/kaggle/working/yolov11n_custom_attention_NhomA/threshold_sweep')
RESULT_CSV = OUTPUT_ROOT / 'a2_threshold_sweep_results.csv'
ERROR_CSV = OUTPUT_ROOT / 'a2_threshold_sweep_errors.csv'

MODULE_KEYS = [
    'baseline_clean',
    'triplet_attention_segment_head',
    'cote_gate',
    'cote_gate_strong_augmentation',
    'lpsc_gate',
    'lpsc_gate_strong_augmentation',
    'cesa_lite_segment_head',
    'sge_eca_head_gate',
]

CHECKPOINT_OVERRIDES = {
    # 'baseline_clean': '/kaggle/working/runs/segment/.../weights/best.pt',
}

IMGSZ = DEFAULT_IMGSZ
CONF_LIST = DEFAULT_CONF_LIST
IOU_LIST = DEFAULT_IOU_LIST
USE_TTA = False  # Keep threshold calibration pure; set True for TTA + threshold sweep after A1.
RUN_YAML_SANITY = True
RUN_VAL_METRICS = False

# Standalone data/prediction controls matching the clean baseline flow.
PREPARE_DATASET = True
DOWNLOAD_DATASET_IF_MISSING = True
FORCE_REDOWNLOAD_DATASET = False
RUN_TEST_PREDICTION_PREVIEW = True
PREDICT_TEST_LIMIT = 8
PREDICT_CONF = 0.10


### Download Data and Leakage-Safe Split


In [ ]:
# ============================================================
# Standalone data preparation: download + leakage-safe split
# Mirrors the clean baseline data setup so this notebook can run alone on Kaggle.
# ============================================================
from pathlib import Path
from collections import defaultdict, Counter
import importlib.util
import os
import random
import re
import shutil
import subprocess
import sys
import yaml

BASE_DATA_DIR = Path(globals().get('BASE_DATA_DIR', '/kaggle/working/shrimpDisHandSegV2-1')).expanduser().resolve()
DATA_YAML = Path(globals().get('DATA_YAML', BASE_DATA_DIR / 'data.yaml')).expanduser().resolve()
SOURCE_DATA_YAML = Path(globals().get('SOURCE_DATA_YAML', DATA_YAML)).expanduser().resolve()

PREPARE_DATASET = bool(globals().get('PREPARE_DATASET', True))
DOWNLOAD_DATASET_IF_MISSING = bool(globals().get('DOWNLOAD_DATASET_IF_MISSING', True))
FORCE_REDOWNLOAD_DATASET = bool(globals().get('FORCE_REDOWNLOAD_DATASET', False))

ROBOFLOW_API_KEY_DIRECT = globals().get('ROBOFLOW_API_KEY_DIRECT', 'KOEk0qLzBFDc7zfyxtgs')
ROBOFLOW_WORKSPACE = globals().get('ROBOFLOW_WORKSPACE', 'lets-try-this')
ROBOFLOW_PROJECT = globals().get('ROBOFLOW_PROJECT', 'shrimpdishandsegv2')
ROBOFLOW_VERSION = int(globals().get('ROBOFLOW_VERSION', 1))
ROBOFLOW_FORMAT = globals().get('ROBOFLOW_FORMAT', 'yolo26')

DATA_PREP_SEED = int(globals().get('DEFAULT_SEED', globals().get('SEED', 42)))
GROUP_SPLIT_BY_SHRIMP = bool(globals().get('GROUP_SPLIT_BY_SHRIMP', True))
GROUP_STRATIFY_BY_DISEASE = bool(globals().get('GROUP_STRATIFY_BY_DISEASE', True))
REBUILD_SPLIT_FROM_ALL_SPLITS = bool(globals().get('REBUILD_SPLIT_FROM_ALL_SPLITS', True))
TRAIN_RATIO = float(globals().get('TRAIN_RATIO', 0.80))
VAL_RATIO = float(globals().get('VAL_RATIO', 0.10))
IMAGE_EXTENSIONS = tuple(globals().get('IMAGE_EXTENSIONS', ('.jpg', '.jpeg', '.png', '.bmp', '.webp')))

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)


def _dataset_has_images(base_path: Path) -> bool:
    for split in ('train', 'valid', 'test'):
        image_dir = base_path / split / 'images'
        if image_dir.exists() and any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in image_dir.iterdir()):
            return True
    return False


def get_roboflow_api_key() -> str:
    direct = str(ROBOFLOW_API_KEY_DIRECT).strip()
    if direct:
        return direct
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key.strip()
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


def download_roboflow_dataset_if_needed() -> Path:
    global BASE_DATA_DIR, DATA_YAML, SOURCE_DATA_YAML

    if not FORCE_REDOWNLOAD_DATASET and DATA_YAML.exists() and _dataset_has_images(BASE_DATA_DIR):
        print(f'Dataset already present: {BASE_DATA_DIR}')
        return BASE_DATA_DIR

    if not DOWNLOAD_DATASET_IF_MISSING:
        raise FileNotFoundError(
            f'Dataset not found at {BASE_DATA_DIR}. Set DOWNLOAD_DATASET_IF_MISSING=True or upload the dataset.'
        )

    if importlib.util.find_spec('roboflow') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

    from roboflow import Roboflow

    api_key = get_roboflow_api_key()
    if not api_key:
        raise RuntimeError(
            'Missing Roboflow API key. Add a Kaggle Secret named ROBOFLOW_API_KEY, '
            'set ROBOFLOW_API_KEY env var, or fill ROBOFLOW_API_KEY_DIRECT in this notebook.'
        )

    if FORCE_REDOWNLOAD_DATASET and BASE_DATA_DIR.exists():
        shutil.rmtree(BASE_DATA_DIR)

    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(ROBOFLOW_FORMAT)
    location = Path(getattr(dataset, 'location', BASE_DATA_DIR)).resolve()
    if location.exists():
        BASE_DATA_DIR = location
        DATA_YAML = BASE_DATA_DIR / 'data.yaml'
        SOURCE_DATA_YAML = DATA_YAML
    print(f'Roboflow dataset ready: {BASE_DATA_DIR}')
    return BASE_DATA_DIR


def normalize_roboflow_stem(stem: str) -> str:
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


def ensure_split_dirs(base_path: Path) -> None:
    for split in ('train', 'valid', 'test'):
        for subdir in ('images', 'labels'):
            (base_path / split / subdir).mkdir(parents=True, exist_ok=True)


def parse_shrimp_group_key(image_name: str):
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{Path(image_name).stem}', 'unparsed', None, None
    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    img_num = int(match.group('img_num'))
    return f'{disease.lower()}::{shrimp_id}', disease, shrimp_id, img_num


def image_files_in_split(base_path: Path, split: str) -> list[Path]:
    image_dir = base_path / split / 'images'
    if not image_dir.exists():
        return []
    return sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def move_image_and_label(base_path: Path, image_path: Path, target_split: str) -> None:
    target_img_dir = base_path / target_split / 'images'
    target_lbl_dir = base_path / target_split / 'labels'
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_name = f'{image_path.stem}.txt'
    label_src = image_path.parent.parent / 'labels' / label_name
    image_dst = target_img_dir / image_path.name
    label_dst = target_lbl_dir / label_name

    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination would be overwritten: {image_dst}')
        shutil.move(str(image_path), str(image_dst))

    if label_src.exists():
        if label_src.resolve() != label_dst.resolve():
            if label_dst.exists():
                raise FileExistsError(f'Duplicate label destination would be overwritten: {label_dst}')
            shutil.move(str(label_src), str(label_dst))
    else:
        label_dst.write_text('', encoding='utf-8')


def rebuild_train_pool_from_all_splits(base_path: Path) -> list[Path]:
    all_images = []
    for split in ('train', 'valid', 'test'):
        all_images.extend(image_files_in_split(base_path, split))
    for image_path in sorted(all_images):
        move_image_and_label(base_path, image_path, 'train')
    return image_files_in_split(base_path, 'train')


def remove_yolo_label_caches(root: Path) -> None:
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')


def disease_for_group(filenames: list[str]) -> str:
    diseases = [parse_shrimp_group_key(filename)[1] for filename in filenames]
    counts = Counter(diseases)
    if len(counts) > 1:
        print(f'Warning: group has mixed disease names: {dict(counts)}')
    return counts.most_common(1)[0][0]


def split_one_stratum(items: list[tuple[str, list[str]]]):
    n = len(items)
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count
    if n >= 3:
        if val_count == 0:
            val_count = 1
            train_count -= 1
        if test_count == 0:
            test_count = 1
            train_count -= 1
    if train_count < 1 and n > 0:
        train_count = 1
    while train_count + val_count + test_count > n:
        train_count -= 1
    test_count = n - train_count - val_count
    return items[:train_count], items[train_count:train_count + val_count], items[train_count + val_count:]


def grouped_stratified_split(group_items: list[tuple[str, list[str]]]):
    strata = defaultdict(list)
    for group_key, filenames in group_items:
        strata[disease_for_group(filenames)].append((group_key, filenames))

    split_to_groups = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(DATA_PREP_SEED)
    for disease, items in sorted(strata.items()):
        items = sorted(items, key=lambda item: item[0])
        rng.shuffle(items)
        train_items, val_items, test_items = split_one_stratum(items)
        split_to_groups['train'].extend(train_items)
        split_to_groups['valid'].extend(val_items)
        split_to_groups['test'].extend(test_items)
        print(
            f'  - {disease}: {len(train_items)} train groups, '
            f'{len(val_items)} valid groups, {len(test_items)} test groups'
        )
    return {split: sorted(items, key=lambda item: item[0]) for split, items in split_to_groups.items()}


def grouped_random_split(group_items: list[tuple[str, list[str]]]):
    group_items = sorted(group_items, key=lambda item: item[0])
    random.Random(DATA_PREP_SEED).shuffle(group_items)
    n_groups = len(group_items)
    train_group_count = int(TRAIN_RATIO * n_groups)
    val_group_count = int(VAL_RATIO * n_groups)
    return {
        'train': group_items[:train_group_count],
        'valid': group_items[train_group_count:train_group_count + val_group_count],
        'test': group_items[train_group_count + val_group_count:],
    }


def split_summary(split_groups: list[tuple[str, list[str]]]):
    group_diseases = Counter()
    image_diseases = Counter()
    for _, filenames in split_groups:
        group_diseases[disease_for_group(filenames)] += 1
        for filename in filenames:
            image_diseases[parse_shrimp_group_key(filename)[1]] += 1
    return group_diseases, image_diseases


def split_grouped_by_shrimp(base_path: Path) -> None:
    image_paths = rebuild_train_pool_from_all_splits(base_path) if REBUILD_SPLIT_FROM_ALL_SPLITS else image_files_in_split(base_path, 'train')
    if not image_paths:
        raise FileNotFoundError(f'No images found under {base_path}. Download/upload the Roboflow dataset first.')

    groups = defaultdict(list)
    disease_counts = Counter()
    unparsed = []
    for image_path in image_paths:
        group_key, disease, _, _ = parse_shrimp_group_key(image_path.name)
        groups[group_key].append(image_path.name)
        disease_counts[disease] += 1
        if disease == 'unparsed':
            unparsed.append(image_path.name)

    group_items = sorted(groups.items(), key=lambda item: item[0])
    if GROUP_STRATIFY_BY_DISEASE:
        print('Building shrimp-grouped, disease-stratified split:')
        split_to_groups = grouped_stratified_split(group_items)
    else:
        print('Building shrimp-grouped random split:')
        split_to_groups = grouped_random_split(group_items)

    for split, split_groups in split_to_groups.items():
        for _, filenames in split_groups:
            for filename in filenames:
                move_image_and_label(base_path, base_path / 'train' / 'images' / filename, split)

    print('Shrimp-grouped split complete:')
    for split, split_groups in split_to_groups.items():
        image_count = sum(len(filenames) for _, filenames in split_groups)
        group_diseases, image_diseases = split_summary(split_groups)
        print(f'  - {split}: {len(split_groups)} shrimp groups, {image_count} images')
        print(f'    group disease counts: {dict(sorted(group_diseases.items()))}')
        print(f'    image disease counts: {dict(sorted(image_diseases.items()))}')

    print('Source filename disease counts before split:', dict(sorted(disease_counts.items())))
    if unparsed:
        print(f'Warning: {len(unparsed)} filenames did not match the shrimp naming pattern.')
        print('First unparsed examples:', unparsed[:10])

    group_to_split = {}
    leakage = []
    for split in ('train', 'valid', 'test'):
        for image_path in image_files_in_split(base_path, split):
            group_key = parse_shrimp_group_key(image_path.name)[0]
            previous_split = group_to_split.setdefault(group_key, split)
            if previous_split != split:
                leakage.append((group_key, previous_split, split, image_path.name))
    if leakage:
        raise RuntimeError(f'Shrimp-level split leakage detected: {leakage[:10]}')
    print('Shrimp-level leakage check passed.')
    remove_yolo_label_caches(base_path)


def split_image_level_if_needed(base_path: Path) -> None:
    valid_images_dir = base_path / 'valid' / 'images'
    test_images_dir = base_path / 'test' / 'images'
    if any(valid_images_dir.glob('*')) or any(test_images_dir.glob('*')):
        print('Existing valid/test split detected. Keeping downloaded split.')
        return
    image_files = sorted(p.name for p in image_files_in_split(base_path, 'train'))
    random.Random(DATA_PREP_SEED).shuffle(image_files)
    train_count = int(0.8 * len(image_files))
    val_count = int(0.1 * len(image_files))
    for filename in image_files[train_count:train_count + val_count]:
        move_image_and_label(base_path, base_path / 'train' / 'images' / filename, 'valid')
    for filename in image_files[train_count + val_count:]:
        move_image_and_label(base_path, base_path / 'train' / 'images' / filename, 'test')
    print('Image-level split complete.')
    remove_yolo_label_caches(base_path)


def update_data_yaml_absolute_paths(base_path: Path) -> Path:
    data_yaml_path = base_path / 'data.yaml'
    if not data_yaml_path.exists():
        raise FileNotFoundError(f'data.yaml not found after dataset preparation: {data_yaml_path}')
    with data_yaml_path.open('r', encoding='utf-8') as f:
        content = yaml.safe_load(f) or {}
    content['train'] = str(base_path / 'train' / 'images')
    content['val'] = str(base_path / 'valid' / 'images')
    content['test'] = str(base_path / 'test' / 'images')
    with data_yaml_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(content, f, sort_keys=False)
    print(f'data.yaml updated with absolute paths: {data_yaml_path}')
    return data_yaml_path


def validate_prepared_dataset(base_path: Path) -> None:
    missing = []
    for split in ('train', 'valid', 'test'):
        for subdir in ('images', 'labels'):
            path = base_path / split / subdir
            if not path.exists():
                missing.append(str(path))
    if missing:
        raise FileNotFoundError('Prepared grouped split is incomplete. Missing: ' + ', '.join(missing))
    counts = {split: len(image_files_in_split(base_path, split)) for split in ('train', 'valid', 'test')}
    if counts['train'] == 0:
        raise RuntimeError(f'No training images found after data preparation: {counts}')
    print('Prepared dataset image counts:', counts)


def prepare_standalone_dataset() -> tuple[Path, Path]:
    global BASE_DATA_DIR, DATA_YAML, SOURCE_DATA_YAML
    BASE_DATA_DIR = download_roboflow_dataset_if_needed()
    ensure_split_dirs(BASE_DATA_DIR)
    if GROUP_SPLIT_BY_SHRIMP:
        split_grouped_by_shrimp(BASE_DATA_DIR)
    else:
        split_image_level_if_needed(BASE_DATA_DIR)
    DATA_YAML = update_data_yaml_absolute_paths(BASE_DATA_DIR)
    SOURCE_DATA_YAML = DATA_YAML
    validate_prepared_dataset(BASE_DATA_DIR)
    globals()['BASE_DATA_DIR'] = BASE_DATA_DIR
    globals()['DATA_YAML'] = DATA_YAML
    globals()['SOURCE_DATA_YAML'] = SOURCE_DATA_YAML
    return BASE_DATA_DIR, DATA_YAML


if PREPARE_DATASET:
    BASE_DATA_DIR, DATA_YAML = prepare_standalone_dataset()
else:
    print('PREPARE_DATASET=False; skipping download/split. Existing BASE_DATA_DIR will be used.')
    DATA_YAML = Path(globals().get('DATA_YAML', BASE_DATA_DIR / 'data.yaml')).expanduser().resolve()
    SOURCE_DATA_YAML = Path(globals().get('SOURCE_DATA_YAML', DATA_YAML)).expanduser().resolve()

print('BASE_DATA_DIR:', BASE_DATA_DIR)
print('DATA_YAML:', DATA_YAML)


### YAML Build and Shape Sanity


In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
checkpoint_df = pd.DataFrame(checkpoint_table(MODULE_KEYS, overrides=CHECKPOINT_OVERRIDES))
display(checkpoint_df)

if RUN_YAML_SANITY:
    sanity_df = pd.DataFrame(sanity_check_candidate_yamls(MODULE_KEYS, imgsz=IMGSZ, device='cpu'))
    display(sanity_df[['module_key', 'status', 'model_yaml', 'output_shapes'] if 'output_shapes' in sanity_df.columns else sanity_df.columns])
    failed = sanity_df[sanity_df['status'] != 'ok'] if not sanity_df.empty else sanity_df
    if not failed.empty:
        raise RuntimeError('Model YAML sanity check failed. Fix build/shape errors before sweeping thresholds.')


### Train or Evaluate


In [ ]:
completed = existing_eval_keys(RESULT_CSV)
planned = [
    (candidate, conf, iou)
    for candidate in selected_candidates(MODULE_KEYS)
    for conf in CONF_LIST
    for iou in IOU_LIST
]
print('Planned evaluations:', len(planned))
print('Already completed rows:', len(completed))

for candidate, conf, iou in planned:
    key = (candidate['key'], round(float(conf), 4), round(float(iou), 4), bool(USE_TTA))
    if key in completed:
        print(f"SKIP existing: {candidate['key']} conf={conf} iou={iou} tta={USE_TTA}")
        continue

    checkpoint = resolve_checkpoint(candidate, overrides=CHECKPOINT_OVERRIDES)
    if checkpoint is None:
        row = {
            'module_key': candidate['key'],
            'display_name': candidate.get('display_name'),
            'conf': conf,
            'iou': iou,
            'tta_augment': USE_TTA,
            'status': 'missing_checkpoint',
            'error': 'No checkpoint found; set CHECKPOINT_OVERRIDES.',
        }
        append_csv_row(ERROR_CSV, row)
        print(f"MISSING checkpoint: {candidate['key']}")
        continue

    try:
        print(f"RUN {candidate['key']} conf={conf} iou={iou} tta={USE_TTA}")
        row = evaluate_checkpoint(
            candidate=candidate,
            checkpoint=checkpoint,
            dataset_dir=BASE_DATA_DIR,
            data_yaml=DATA_YAML,
            output_root=OUTPUT_ROOT,
            imgsz=IMGSZ,
            conf=float(conf),
            iou=float(iou),
            augment=USE_TTA,
            run_val=RUN_VAL_METRICS,
            plots=False,
        )
        append_csv_row(RESULT_CSV, row)
        completed.add(key)
        print(
            f"OK {candidate['key']} conf={conf} iou={iou}: "
            f"score={row['healthy_aware_score']:.4f}, "
            f"labeled_mAP50={row['labeled_test_mask_map50']:.4f}, "
            f"healthy_FP={row['healthy_test_healthy_mask_fp_rate']:.4f}"
        )
    except Exception as exc:
        row = {
            'module_key': candidate['key'],
            'display_name': candidate.get('display_name'),
            'conf': conf,
            'iou': iou,
            'tta_augment': USE_TTA,
            'status': 'error',
            'error_type': exc.__class__.__name__,
            'error': str(exc),
        }
        append_csv_row(ERROR_CSV, row)
        print(f"ERROR {candidate['key']} conf={conf} iou={iou}: {exc.__class__.__name__}: {exc}")

print('Saved:', RESULT_CSV)
print('Errors:', ERROR_CSV)

# Standalone data/prediction controls matching the clean baseline flow.
PREPARE_DATASET = True
DOWNLOAD_DATASET_IF_MISSING = True
FORCE_REDOWNLOAD_DATASET = False
RUN_TEST_PREDICTION_PREVIEW = True
PREDICT_TEST_LIMIT = 8
PREDICT_CONF = 0.10


### Inspect Metrics and Reports


In [ ]:
if not RESULT_CSV.exists():
    raise FileNotFoundError(f'No sweep result CSV yet: {RESULT_CSV}')

df = pd.read_csv(RESULT_CSV)
rank_cols = [
    'module_key', 'augmentation_policy', 'tta_augment', 'conf', 'iou',
    'healthy_aware_score', 'labeled_test_mask_map50', 'labeled_test_mask_map50_95',
    'full_test_mask_map50', 'full_test_mask_map50_95',
    'healthy_test_healthy_mask_fp_rate', 'healthy_test_healthy_fp_masks_per_image',
    'labeled_test_disease_box_miss_rate', 'labeled_test_mask_count_mae',
]
rank_cols = [c for c in rank_cols if c in df.columns]

best_by_module = (
    df.sort_values('healthy_aware_score', ascending=False)
      .groupby('module_key', as_index=False)
      .head(1)[rank_cols]
      .sort_values('healthy_aware_score', ascending=False)
      .reset_index(drop=True)
)

best_csv = OUTPUT_ROOT / 'a2_threshold_best_by_module.csv'
best_by_module.to_csv(best_csv, index=False)
print('Saved best-by-module:', best_csv)
display(best_by_module)


### Test Set Prediction Preview


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

# Standalone test-set prediction preview.
# Runs after training/evaluation and mirrors the clean baseline visual inspection cell.
from pathlib import Path
import math
import pandas as pd
from IPython.display import display

RUN_TEST_PREDICTION_PREVIEW = bool(globals().get('RUN_TEST_PREDICTION_PREVIEW', True))
PREDICT_TEST_LIMIT = int(globals().get('PREDICT_TEST_LIMIT', 8))
PREDICT_CONF = float(globals().get('PREDICT_CONF', globals().get('CONF', 0.10)))
PREDICT_IOU = float(globals().get('PREDICT_IOU', globals().get('IOU', 0.70)))
PREDICT_IMGSZ = int(globals().get('PREDICT_IMGSZ', globals().get('IMGSZ', globals().get('DEFAULT_IMGSZ', 640))))
PREDICT_SAVE = bool(globals().get('PREDICT_SAVE', True))

IMAGE_EXTENSIONS = tuple(globals().get('IMAGE_EXTENSIONS', ('.jpg', '.jpeg', '.png', '.bmp', '.webp')))


def _as_path(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none'}:
        return None
    return Path(text).expanduser()


def _existing_path(value):
    path = _as_path(value)
    if path is not None and path.exists():
        return path
    try:
        resolver = globals().get('resolve_existing_path')
        if callable(resolver):
            resolved = resolver(value)
            if resolved is not None and Path(resolved).exists():
                return Path(resolved)
    except Exception:
        pass
    return None


def _read_csv_if_exists(path_value):
    path = _existing_path(path_value)
    if path is None:
        return None
    try:
        df = pd.read_csv(path)
        if df.empty:
            return None
        df['_source_csv'] = str(path)
        return df
    except Exception as exc:
        print(f'Could not read CSV {path}: {exc}')
        return None


def _candidate_result_tables():
    tables = []
    for var_name in ('RESULT_CSV', 'BEST_CSV'):
        if var_name in globals():
            df = _read_csv_if_exists(globals()[var_name])
            if df is not None:
                tables.append(df)

    if 'OUTPUT_ROOT' in globals() and 'EXPERIMENT_KEY' in globals():
        report_dir = Path(globals()['OUTPUT_ROOT']) / str(globals()['EXPERIMENT_KEY']) / 'reports'
        for name in (f"{globals()['EXPERIMENT_KEY']}_eval_results.csv", f"{globals()['EXPERIMENT_KEY']}_train_runs.csv"):
            df = _read_csv_if_exists(report_dir / name)
            if df is not None:
                tables.append(df)

    if isinstance(globals().get('rows'), list) and globals()['rows']:
        df = pd.DataFrame(globals()['rows'])
        df['_source_csv'] = 'in_memory_rows'
        tables.append(df)

    if isinstance(globals().get('row'), dict) and globals()['row']:
        df = pd.DataFrame([globals()['row']])
        df['_source_csv'] = 'in_memory_row'
        tables.append(df)

    direct_rows = []
    if _existing_path(globals().get('best_path')) is not None:
        direct_rows.append({
            'checkpoint': str(_existing_path(globals().get('best_path'))),
            'dataset_dir': str(globals().get('DATASET_SNAPSHOT', globals().get('BASE_DATA_DIR', ''))),
            'data_yaml': str(globals().get('TRAIN_DATA_YAML', globals().get('DATA_YAML', globals().get('SOURCE_DATA_YAML', '')))),
            'module_key': globals().get('RUN_NAME', 'selected_run'),
            '_source_csv': 'best_path_variable',
        })
    if direct_rows:
        tables.append(pd.DataFrame(direct_rows))

    return tables


def _select_prediction_row():
    rows_out = []
    for table in _candidate_result_tables():
        for _, row_item in table.iterrows():
            checkpoint = _existing_path(row_item.get('checkpoint')) or _existing_path(row_item.get('best_checkpoint'))
            if checkpoint is None:
                continue
            item = row_item.to_dict()
            item['_checkpoint_path'] = str(checkpoint)
            dataset = _existing_path(item.get('dataset_dir')) or _existing_path(globals().get('BASE_DATA_DIR'))
            data_yaml = _existing_path(item.get('data_yaml')) or _existing_path(globals().get('DATA_YAML')) or _existing_path(globals().get('SOURCE_DATA_YAML'))
            if dataset is not None:
                item['_dataset_dir'] = str(dataset)
            if data_yaml is not None:
                item['_data_yaml'] = str(data_yaml)
            rows_out.append(item)

    if not rows_out:
        return None, pd.DataFrame()

    df = pd.DataFrame(rows_out)
    score_cols = [
        'healthy_aware_score',
        'healthy_aware_labeled_test_mask_map50',
        'labeled_test_mask_map50',
        'full_test_mask_map50',
        'best_fitness',
    ]
    for col in score_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    sort_cols = [c for c in score_cols if c in df.columns and df[c].notna().any()]
    if sort_cols:
        df = df.sort_values(sort_cols, ascending=[False] * len(sort_cols), na_position='last')
    return df.iloc[0].to_dict(), df


def _image_files(image_dir):
    image_dir = Path(image_dir)
    files = []
    for ext in IMAGE_EXTENSIONS:
        files.extend(image_dir.glob(f'*{ext}'))
        files.extend(image_dir.glob(f'*{ext.upper()}'))
    return sorted(set(files))


def _has_nonempty_label(dataset_dir, image_path):
    label_path = Path(dataset_dir) / 'test' / 'labels' / f'{Path(image_path).stem}.txt'
    return label_path.exists() and bool(label_path.read_text(encoding='utf-8').strip())


def _select_test_images(dataset_dir, limit):
    test_image_dir = Path(dataset_dir) / 'test' / 'images'
    if not test_image_dir.exists():
        return [], []
    all_images = _image_files(test_image_dir)
    labeled = [p for p in all_images if _has_nonempty_label(dataset_dir, p)]
    healthy = [p for p in all_images if not _has_nonempty_label(dataset_dir, p)]
    selected_labeled = labeled[:limit]
    selected_healthy = healthy[:limit]
    if not selected_labeled and all_images:
        selected_labeled = all_images[:limit]
    return selected_labeled, selected_healthy


def _prediction_summary_rows(results, image_paths, split_label):
    summary = []
    for image_path, result in zip(image_paths, results):
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        max_conf = None
        if result.boxes is not None and getattr(result.boxes, 'conf', None) is not None and len(result.boxes.conf):
            max_conf = float(result.boxes.conf.max().detach().cpu())
        summary.append({
            'subset': split_label,
            'image': Path(image_path).name,
            'boxes': box_count,
            'masks': mask_count,
            'max_conf': max_conf,
        })
    return summary


def _show_predictions(results, image_paths, title_prefix):
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f'Matplotlib is not available; predictions were saved but not displayed: {exc}')
        return
    for image_path, result in zip(image_paths, results):
        plt.figure(figsize=(8, 8))
        plt.imshow(result.plot())
        plt.title(f'{title_prefix}: {Path(image_path).name}')
        plt.axis('off')
        plt.show()


if RUN_TEST_PREDICTION_PREVIEW:
    selected_row, candidate_df = _select_prediction_row()
    if selected_row is None:
        print('No trained/evaluated checkpoint found for test prediction preview yet.')
        print('Run the training/evaluation cells above, or fill CHECKPOINT_OVERRIDES for inference notebooks.')
    else:
        display_cols = [c for c in [
            'module_key', 'variant_module_key', 'experiment_key', 'seed',
            'healthy_aware_score', 'labeled_test_mask_map50', 'full_test_mask_map50',
            '_checkpoint_path', '_dataset_dir', '_source_csv'
        ] if c in candidate_df.columns]
        print('Selected checkpoint for test prediction preview:')
        display(candidate_df[display_cols].head(10) if display_cols else candidate_df.head(10))

        checkpoint_path = Path(selected_row['_checkpoint_path'])
        dataset_dir = Path(selected_row.get('_dataset_dir', globals().get('BASE_DATA_DIR', '/kaggle/working/shrimpDisHandSegV2-1')))
        labeled_images, healthy_images = _select_test_images(dataset_dir, PREDICT_TEST_LIMIT)
        if not labeled_images and not healthy_images:
            print(f'No test images found under {dataset_dir / "test" / "images"}.')
        else:
            register_fn = globals().get('register_group_d_modules') or globals().get('register_all_custom_attention_modules')
            if callable(register_fn):
                register_fn()
            from ultralytics import YOLO

            model_preview = YOLO(str(checkpoint_path))
            output_root = Path(globals().get('OUTPUT_ROOT', '/kaggle/working'))
            predict_dir = output_root / 'test_prediction_preview'
            predict_dir.mkdir(parents=True, exist_ok=True)

            summary_rows = []
            if labeled_images:
                labeled_results = model_preview.predict(
                    source=[str(p) for p in labeled_images],
                    imgsz=PREDICT_IMGSZ,
                    conf=PREDICT_CONF,
                    iou=PREDICT_IOU,
                    save=PREDICT_SAVE,
                    project=str(predict_dir),
                    name='labeled_test',
                    exist_ok=True,
                    verbose=False,
                )
                summary_rows.extend(_prediction_summary_rows(labeled_results, labeled_images, 'labeled_test'))
                _show_predictions(labeled_results, labeled_images, 'Labeled test prediction')

            if healthy_images:
                healthy_results = model_preview.predict(
                    source=[str(p) for p in healthy_images],
                    imgsz=PREDICT_IMGSZ,
                    conf=PREDICT_CONF,
                    iou=PREDICT_IOU,
                    save=PREDICT_SAVE,
                    project=str(predict_dir),
                    name='healthy_test',
                    exist_ok=True,
                    verbose=False,
                )
                summary_rows.extend(_prediction_summary_rows(healthy_results, healthy_images, 'healthy_test'))
                _show_predictions(healthy_results, healthy_images, 'Healthy test prediction')

            prediction_summary = pd.DataFrame(summary_rows)
            prediction_csv = predict_dir / 'test_prediction_preview_summary.csv'
            prediction_summary.to_csv(prediction_csv, index=False)
            print('Saved test prediction preview directory:', predict_dir)
            print('Saved prediction summary:', prediction_csv)
            display(prediction_summary)
else:
    print('RUN_TEST_PREDICTION_PREVIEW=False; skipping test prediction preview.')
